In [159]:
################################################# CLAUDE ###########################################
import pandas as pd
import numpy as np
from typing import Iterable, Optional, Dict, Any, List, Set, Tuple
from collections import defaultdict
import zipfile
import pyranges as pr



def _normalize_chr(s: pd.Series) -> pd.Series:
    s = s.astype(str)
    return s.str.replace(r"^(?!(chr))", "chrom", regex=True)



# =============================================================================
# SCHEMA HELPERS
# =============================================================================

def empty_assay_entry() -> Dict[str, Any]:
    return {"score": None, "p_value": None, "strength": "none"}


def empty_biosample_assays(assay_types: List[str]) -> Dict[str, Dict[str, Any]]:
    return {assay: empty_assay_entry() for assay in assay_types}


def empty_conservation_entry() -> Dict[str, int]:
    return {"n_biosamples": 0, "n_strong": 0, "n_weak": 0}


def empty_conservation_block(assay_types: List[str]) -> Dict[str, Dict[str, int]]:
    return {assay: empty_conservation_entry() for assay in assay_types}


def empty_screen_block(biosamples: List[str], assay_types: List[str]) -> Dict[str, Any]:
    return {
        "per_biosample": {bio: empty_biosample_assays(assay_types) for bio in biosamples},
        "conservation_global": empty_conservation_block(assay_types),
        "conservation_breast": empty_conservation_block(assay_types),
    }


def ensure_screen_block(
    val: Any,
    biosamples: List[str],
    assay_types: List[str],
) -> Dict[str, Any]:
    """Coerce any value into a valid screen block, filling missing keys."""
    if not isinstance(val, dict):
        return empty_screen_block(biosamples, assay_types)

    val.setdefault("per_biosample", {})
    val.setdefault("conservation_global", {})
    val.setdefault("conservation_breast", {})

    # Ensure all biosamples and assays exist
    for bio in biosamples:
        if bio not in val["per_biosample"] or not isinstance(val["per_biosample"][bio], dict):
            val["per_biosample"][bio] = empty_biosample_assays(assay_types)
        else:
            for assay in assay_types:
                val["per_biosample"][bio].setdefault(assay, empty_assay_entry())

    # Ensure conservation has all assays
    for cons_key in ["conservation_global", "conservation_breast"]:
        if not isinstance(val[cons_key], dict):
            val[cons_key] = empty_conservation_block(assay_types)
        else:
            for assay in assay_types:
                val[cons_key].setdefault(assay, empty_conservation_entry())

    return val


def empty_abc_celltype_entry() -> Dict[str, Any]:
    return {
        "ABC_score": None,
        "ABC_num": None,
        "activity": None,
        "distance": None,
        "element_class": None,
        "is_self_promoter": False,
        "hic_pl_scaled": None,
        "powerlaw_score": None,
        "gene_expr": None,
        "promoter_activity_q": None,
        "gene_is_expressed": False,
        "rank_within_gene": None,
        "is_present": False,
        "is_strong": False,
    }


def empty_abc_block(celltypes: List[str]) -> Dict[str, Dict[str, Any]]:
    return {ct: empty_abc_celltype_entry() for ct in celltypes}


# =============================================================================
# COLUMN DEFINITIONS & RENAMING
# =============================================================================

COLS_SCREEN_EXP = [
    "cCRE ID", "Gene ID", "Common Gene Name", "Gene Type",
    "Assay Type", "Experiment ID", "Biosample", "Score", "P-value",
]

COLS_SCREEN_COMP = [
    "cCRE ID", "Gene ID", "Common Gene Name", "Gene Type",
    "Assay Type", "Region", "Experiment ID", "Biosample", "Score",
]

COLS_ABC_RAW = [
    "chr", "start", "end", "name", "class", "activity_base",
    "TargetGene", "TargetGeneTSS", "TargetGeneExpression",
    "TargetGenePromoterActivityQuantile", "TargetGeneIsExpressed",
    "distance", "isSelfPromoter", "hic_contact", "powerlaw_contact",
    "powerlaw_contact_reference", "hic_contact_pl_scaled", "hic_pseudocount",
    "hic_contact_pl_scaled_adj", "ABC.Score.Numerator", "ABC.Score",
    "powerlaw.Score.Numerator", "powerlaw.Score", "CellType",
]

RENAME_SCREEN = {
    "cCRE ID": "ENCODE_id",
    "Gene ID": "gene_id",
    "Common Gene Name": "gene_name",
    "Gene Type": "gene_type",
    "Assay Type": "assay_type",
    "Experiment ID": "experiment_id",
    "Biosample": "biosample",
    "Score": "score",
    "P-value": "p_value",
    "Region": "region",
}

RENAME_ABC = {
    "name": "enh_id",
    "chr": "chrom",
    "class": "element_class",
    "activity_base": "activity",
    "TargetGene": "gene_name",
    "TargetGeneExpression": "gene_expr",
    "TargetGenePromoterActivityQuantile": "promoter_activity_q",
    "TargetGeneIsExpressed": "gene_is_expressed",
    "isSelfPromoter": "is_self_promoter",
    "hic_contact_pl_scaled_adj": "hic_pl_scaled",
    "ABC.Score.Numerator": "ABC_num",
    "ABC.Score": "ABC_score",
    "powerlaw.Score": "powerlaw_score",
    "CellType": "cell_type",
}

LINK_COLS = ["ENCODE_id", "gene_id", "gene_name"]

ABC_NUMERIC_COLS = [
    "start", "end", "activity", "gene_expr", "promoter_activity_q",
    "distance", "hic_pl_scaled", "ABC_num", "ABC_score", "powerlaw_score"
]


# =============================================================================
# STREAMING & FILTERING
# =============================================================================

def stream_and_filter_genes(
    file_handle,
    gene_set: Set[str],
    columns: List[str],
    chunksize: int,
    sep: str = "\t",
    compression: Optional[str] = None,
    add_dummy_pvalue: bool = False,
) -> pd.DataFrame:
    """Stream a tabular file and retain only rows matching the gene set."""
    chunks = []
    reader = pd.read_csv(
        file_handle,
        sep=sep,
        header=None,
        names=columns,
        chunksize=chunksize,
        low_memory=False,
        compression=compression,
    )

    for i, chunk in enumerate(reader):
        chunk["Common Gene Name"] = chunk["Common Gene Name"].astype(str).str.strip()
        chunk["Score"] = pd.to_numeric(chunk["Score"], errors="coerce")

        if i == 10:
            break

        if add_dummy_pvalue:
            chunk["P-value"] = np.nan

        mask = chunk["Common Gene Name"].isin(gene_set)
        if mask.any():
            chunks.append(chunk.loc[mask])

        if (i + 1) % 10 == 0:
            print(f"  Chunk {i + 1}: kept {mask.sum()} rows")

    if not chunks:
        raise ValueError("No rows found for selected genes.")

    df = pd.concat(chunks, ignore_index=True)
    print(f"  Total rows extracted: {len(df)}")
    return df


def stream_and_filter_abc(
    file_path: str,
    gene_set: Set[str],
    celltype_set: Optional[Set[str]],
    keep_cols: List[str],
    chunksize: int,
) -> pd.DataFrame:
    """Stream ABC file and retain rows matching genes and cell types."""
    chunks = []
    reader = pd.read_csv(
        file_path, sep="\t", header=0, names=COLS_ABC_RAW,
        chunksize=chunksize, low_memory=False, compression="infer",
    )

    for i, chunk in enumerate(reader):
        chunk["TargetGene"] = chunk["TargetGene"].astype(str).str.strip()
        chunk["CellType"] = chunk["CellType"].astype(str).str.strip()
        chunk.rename(columns={"chr": "chrom"}, inplace=True)

        mask = chunk["TargetGene"].isin(gene_set)
        if celltype_set:
            mask &= chunk["CellType"].isin(celltype_set)

        if mask.any():
            chunks.append(chunk.loc[mask, keep_cols].copy())

        if (i + 1) % 10 == 0:
            print(f"  Chunk {i + 1}: kept {mask.sum()} rows")

    if not chunks:
        raise ValueError("No ABC rows found for selected genes/cell types.")

    df = pd.concat(chunks, ignore_index=True)
    print(f"  Total rows extracted: {len(df)}")
    return df


# =============================================================================
# STRENGTH CLASSIFICATION
# =============================================================================

def classify_strength(is_strong: bool, is_weak: bool) -> str:
    if is_strong:
        return "strong"
    return "weak" if is_weak else "none"


def apply_strength_classification(
    df: pd.DataFrame,
    weak_quantile: float,
    strong_quantile: float,
    intact_pvalue_threshold: Optional[float] = None,
) -> pd.DataFrame:
    """Compute per-assay quantiles and assign strength labels."""
    quantiles = (
        df.groupby("assay_type")["score"]
        .quantile([weak_quantile, strong_quantile])
        .unstack()
        .rename(columns={weak_quantile: "q_weak", strong_quantile: "q_strong"})
    )
    print("  Per-assay score quantiles:")
    for line in quantiles.to_string().split('\n'):
        print(f"    {line}")

    df = df.merge(quantiles, left_on="assay_type", right_index=True, how="left")

    df["is_strong"] = df["score"] >= df["q_strong"]
    df["is_weak"] = (df["score"] >= df["q_weak"]) & ~df["is_strong"]

    if intact_pvalue_threshold is not None:
        mask = df["assay_type"] == "Intact-HiC"
        df.loc[mask, "is_strong"] &= df.loc[mask, "p_value"] <= intact_pvalue_threshold

    df["strength"] = [classify_strength(s, w) for s, w in zip(df["is_strong"], df["is_weak"])]
    return df


# =============================================================================
# AGGREGATION HELPERS
# =============================================================================

def aggregate_screen_per_link_biosample_assay(
    df: pd.DataFrame,
    link_cols: List[str],
    include_region: bool = False,
) -> pd.DataFrame:
    """Aggregate SCREEN data to one row per (link, biosample, assay_type)."""
    group_cols = link_cols + ["biosample", "assay_type"]

    agg_spec = {
        "score": ("score", "max"),
        "p_value": ("p_value", "min"),
        "any_strong": ("is_strong", "max"),
        "any_weak": ("is_weak", "max"),
        "gene_type": ("gene_type", "first"),
    }
    if include_region and "region" in df.columns:
        agg_spec["region"] = ("region", "first")

    result = df.groupby(group_cols).agg(**agg_spec).reset_index()
    result["strength"] = [
        classify_strength(s, w) for s, w in zip(result["any_strong"], result["any_weak"])
    ]
    return result


def aggregate_abc_per_enhancer_gene_celltype(df: pd.DataFrame) -> pd.DataFrame:
    """Aggregate ABC data to one row per (enhancer, gene, cell_type)."""
    group_cols = ["chrom", "start", "end", "enh_id", "gene_name", "cell_type"]

    return df.groupby(group_cols).agg(
        element_class=("element_class", "first"),
        activity=("activity", "max"),
        distance=("distance", "median"),
        is_self_promoter=("is_self_promoter", "max"),
        hic_pl_scaled=("hic_pl_scaled", "max"),
        ABC_num=("ABC_num", "max"),
        ABC_score=("ABC_score", "max"),
        powerlaw_score=("powerlaw_score", "max"),
        gene_expr=("gene_expr", "first"),
        promoter_activity_q=("promoter_activity_q", "first"),
        gene_is_expressed=("gene_is_expressed", "max"),
    ).reset_index()


def aggregate_abc_to_full_dict(
    df_agg: pd.DataFrame,
    present_threshold: float,
    strong_threshold: float,
) -> pd.DataFrame:
    """
    Add presence/strength flags and collapse to ABC_full dict per (enhancer, gene).
    
    Returns DataFrame with: chr, start, end, enh_id, gene_name, ABC_full
    """
    # Compute flags
    df_agg = df_agg.copy()
    df_agg["is_present"] = df_agg["ABC_score"] >= present_threshold
    df_agg["is_strong"] = (df_agg["ABC_score"] >= strong_threshold) & (df_agg["element_class"] != "promoter")

    # Rank within gene/cell
    max_score = df_agg.groupby(["gene_name", "cell_type"])["ABC_score"].transform("max")
    df_agg["rank_within_gene"] = np.where(max_score > 0, df_agg["ABC_score"] / max_score, 0.0)

    def build_abc_full(df_sub: pd.DataFrame) -> Dict[str, Dict[str, Any]]:
        out = {}
        for _, r in df_sub.iterrows():
            out[r["cell_type"]] = {
                "ABC_score": float(r["ABC_score"]) if pd.notna(r["ABC_score"]) else None,
                "ABC_num": float(r["ABC_num"]) if pd.notna(r["ABC_num"]) else None,
                "activity": float(r["activity"]) if pd.notna(r["activity"]) else None,
                "distance": int(r["distance"]) if pd.notna(r["distance"]) else None,
                "element_class": r["element_class"],
                "is_self_promoter": bool(r["is_self_promoter"]),
                "hic_pl_scaled": float(r["hic_pl_scaled"]) if pd.notna(r["hic_pl_scaled"]) else None,
                "powerlaw_score": float(r["powerlaw_score"]) if pd.notna(r["powerlaw_score"]) else None,
                "gene_expr": float(r["gene_expr"]) if pd.notna(r["gene_expr"]) else None,
                "promoter_activity_q": float(r["promoter_activity_q"]) if pd.notna(r["promoter_activity_q"]) else None,
                "gene_is_expressed": bool(r["gene_is_expressed"]),
                "rank_within_gene": float(r["rank_within_gene"]) if pd.notna(r["rank_within_gene"]) else None,
                "is_present": bool(r["is_present"]),
                "is_strong": bool(r["is_strong"]),
            }
        return out

    return (
        df_agg.groupby(["chrom", "start", "end", "enh_id", "gene_name"], sort=False)
        .apply(build_abc_full)
        .rename("ABC_full")
        .reset_index()
    )


# =============================================================================
# DICT BUILDERS FOR SCREEN DATA
# =============================================================================

def build_biosample_columns(
    df_aggregated: pd.DataFrame,
    df_links: pd.DataFrame,
    link_cols: List[str],
    biosamples: List[str],
    assay_types: List[str],
) -> pd.DataFrame:
    """Add one column per biosample with nested {assay: {score, p_value, strength}}."""

    def make_assay_dict(df_sub: pd.DataFrame) -> Dict[str, Dict[str, Any]]:
        rows_by_assay = {r["assay_type"]: r for _, r in df_sub.iterrows()}
        out = {}
        for assay in assay_types:
            if assay in rows_by_assay:
                r = rows_by_assay[assay]
                out[assay] = {
                    "score": float(r["score"]) if pd.notna(r["score"]) else None,
                    "p_value": float(r["p_value"]) if pd.notna(r["p_value"]) else None,
                    "strength": r["strength"],
                }
            else:
                out[assay] = empty_assay_entry()
        return out

    default = empty_biosample_assays(assay_types)

    for bio in biosamples:
        df_bio = df_aggregated[df_aggregated["biosample"] == bio]
        if df_bio.empty:
            df_links[bio] = [default.copy() for _ in range(len(df_links))]
        else:
            bio_series = df_bio.groupby(link_cols).apply(make_assay_dict)
            df_links = df_links.merge(bio_series.rename(bio).reset_index(), on=link_cols, how="left")
            df_links[bio] = df_links[bio].apply(lambda x: x if isinstance(x, dict) else default.copy())

    return df_links


def build_conservation_column(
    df_aggregated: pd.DataFrame,
    link_cols: List[str],
    biosample_set: Set[str],
    assay_types: List[str],
    column_name: str,
) -> pd.DataFrame:
    """Build conservation summary: {assay_type: {n_biosamples, n_strong, n_weak}}."""
    filtered = df_aggregated[df_aggregated["biosample"].isin(biosample_set)]

    if filtered.empty:
        return None

    cons = (
        filtered.groupby(link_cols + ["assay_type"])
        .agg(
            n_biosamples=("biosample", "nunique"),
            n_strong=("any_strong", "sum"),
            n_weak=("any_weak", "sum"),
        )
        .reset_index()
    )

    def to_dict(df_sub: pd.DataFrame) -> Dict[str, Dict[str, int]]:
        result = {r["assay_type"]: {
            "n_biosamples": int(r["n_biosamples"]),
            "n_strong": int(r["n_strong"]),
            "n_weak": int(r["n_weak"]),
        } for _, r in df_sub.iterrows()}
        
        # Fill missing assays
        for assay in assay_types:
            if assay not in result:
                result[assay] = empty_conservation_entry()
        return result

    return cons.groupby(link_cols).apply(to_dict).rename(column_name).reset_index()


def attach_conservation_columns(
    df_links: pd.DataFrame,
    df_aggregated: pd.DataFrame,
    link_cols: List[str],
    assay_types: List[str],
    global_biosamples: Optional[Set[str]],
    breast_biosamples: Set[str],
) -> pd.DataFrame:
    """Attach conservation_global and conservation_breast columns."""
    default_cons = empty_conservation_block(assay_types)

    # Global conservation
    global_subset = df_aggregated if global_biosamples is None else \
        df_aggregated[df_aggregated["biosample"].isin(global_biosamples)]
    
    cons_global = build_conservation_column(
        global_subset, link_cols, set(global_subset["biosample"]), assay_types, "conservation_global"
    )
    if cons_global is not None:
        df_links = df_links.merge(cons_global, on=link_cols, how="left")
    
    df_links["conservation_global"] = df_links.get("conservation_global", pd.Series([None] * len(df_links))).apply(
        lambda x: x if isinstance(x, dict) else default_cons.copy()
    )

    # Breast conservation
    cons_breast = build_conservation_column(
        df_aggregated, link_cols, breast_biosamples, assay_types, "conservation_breast"
    )
    if cons_breast is not None:
        df_links = df_links.merge(cons_breast, on=link_cols, how="left")
    
    df_links["conservation_breast"] = df_links.get("conservation_breast", pd.Series([None] * len(df_links))).apply(
        lambda x: x if isinstance(x, dict) else default_cons.copy()
    )

    return df_links


# =============================================================================
# SCREEN LINK TABLE BUILDERS
# =============================================================================

def build_screen_exp_links(
    zip_path: str,
    inner_file: str,
    gene_list: Iterable[str],
    panel_biosamples: List[str],
    breast_biosamples: Optional[List[str]] = None,
    global_biosamples: Optional[List[str]] = None,
    weak_quantile: float = 0.90,
    strong_quantile: float = 0.95,
    intact_pvalue_threshold: Optional[float] = None,
    chunksize: int = 300_000,
) -> Tuple[pd.DataFrame, List[str]]:
    """
    Build enhancer-gene links from ENCODE SCREEN 3D-Chromatin (experimental) data.
    
    Returns: (links_dataframe, assay_types_list)
    """
    print("Building SCREEN experimental links...")

    gene_set = set(gene_list)
    breast_set = set(breast_biosamples or panel_biosamples)
    global_set = set(global_biosamples) if global_biosamples else None

    # Stream and filter
    with zipfile.ZipFile(zip_path) as z:
        with z.open(inner_file) as f:
            df = stream_and_filter_genes(f, gene_set, COLS_SCREEN_EXP, chunksize)

    df = df.rename(columns=RENAME_SCREEN)
    df["p_value"] = pd.to_numeric(df["p_value"], errors="coerce")

    assay_types = sorted(df["assay_type"].dropna().unique())
    print(f"  Assay types found: {assay_types}")

    # Strength classification
    df = apply_strength_classification(df, weak_quantile, strong_quantile, intact_pvalue_threshold)

    # Aggregate
    df_agg = aggregate_screen_per_link_biosample_assay(df, LINK_COLS)

    # Base links
    links = df_agg[LINK_COLS + ["gene_type"]].drop_duplicates().reset_index(drop=True)

    # Biosample columns
    links = build_biosample_columns(df_agg, links, LINK_COLS, panel_biosamples, assay_types)

    # Conservation
    links = attach_conservation_columns(links, df_agg, LINK_COLS, assay_types, global_set, breast_set)

    print(f"  Built {len(links)} links")
    return links, assay_types


def build_screen_comp_links(
    gz_path: str,
    gene_list: Iterable[str],
    panel_biosamples: List[str],
    breast_biosamples: Optional[List[str]] = None,
    global_biosamples: Optional[List[str]] = None,
    weak_quantile: float = 0.90,
    strong_quantile: float = 0.95,
    chunksize: int = 300_000,
) -> Tuple[pd.DataFrame, List[str]]:
    """
    Build enhancer-gene links from ENCODE SCREEN Computational-Methods data.
    
    Returns: (links_dataframe, assay_types_list)
    """
    print("Building SCREEN computational links...")

    gene_set = set(gene_list)
    breast_set = set(breast_biosamples or panel_biosamples)
    global_set = set(global_biosamples) if global_biosamples else None

    # Stream and filter
    df = stream_and_filter_genes(
        gz_path, gene_set, COLS_SCREEN_COMP, chunksize,
        compression="gzip", add_dummy_pvalue=True
    )

    df = df.rename(columns=RENAME_SCREEN)
    df["p_value"] = pd.to_numeric(df["p_value"], errors="coerce")

    assay_types = sorted(df["assay_type"].dropna().unique())
    print(f"  Assay types found: {assay_types}")

    # Strength classification
    df = apply_strength_classification(df, weak_quantile, strong_quantile)

    # Aggregate
    df_agg = aggregate_screen_per_link_biosample_assay(df, LINK_COLS, include_region=True)

    # Base links
    links = df_agg[LINK_COLS + ["gene_type"]].drop_duplicates().reset_index(drop=True)

    # Region
    region_series = df_agg.groupby(LINK_COLS)["region"].first().reset_index()
    links = links.merge(region_series, on=LINK_COLS, how="left")

    # Biosample columns
    links = build_biosample_columns(df_agg, links, LINK_COLS, panel_biosamples, assay_types)

    # Conservation
    links = attach_conservation_columns(links, df_agg, LINK_COLS, assay_types, global_set, breast_set)

    print(f"  Built {len(links)} links")
    return links, assay_types


def collapse_screen_to_nested(
    df_links: pd.DataFrame,
    biosamples: List[str],
    assay_types: List[str],
    target_column: str,
) -> pd.DataFrame:
    """Collapse biosample columns into a single nested dict column."""

    def row_to_nested(row) -> Dict[str, Any]:
        per_bio = {}
        for bio in biosamples:
            if bio in row.index and isinstance(row[bio], dict):
                per_bio[bio] = row[bio]
            else:
                per_bio[bio] = empty_biosample_assays(assay_types)

        cons_global = row.get("conservation_global")
        cons_breast = row.get("conservation_breast")

        return {
            "per_biosample": per_bio,
            "conservation_global": cons_global if isinstance(cons_global, dict) else empty_conservation_block(assay_types),
            "conservation_breast": cons_breast if isinstance(cons_breast, dict) else empty_conservation_block(assay_types),
        }

    df_links[target_column] = df_links.apply(row_to_nested, axis=1)

    drop_cols = [c for c in biosamples + ["conservation_global", "conservation_breast"] if c in df_links.columns]
    return df_links.drop(columns=drop_cols)


# =============================================================================
# ABC LINK TABLE BUILDERS
# =============================================================================

def build_abc_enhancer_links(
    abc_path: str,
    gene_list: Iterable[str],
    celltypes: Optional[List[str]] = None,
    present_threshold: float = 0.015,
    strong_threshold: float = 0.05,
    chunksize: int = 500_000,
) -> pd.DataFrame:
    """
    Build enhancer-gene links from ABC model predictions.
    
    Returns DataFrame with columns: chrom, start, end, enh_id, gene_name, ABC_full
    where ABC_full is {cell_type: {...metrics...}}
    """
    print("Building ABC enhancer links...")

    gene_set = set(gene_list)
    celltype_set = set(celltypes) if celltypes else None

    keep_cols = [
        "chrom", "start", "end", "name", "class", "activity_base", "TargetGene",
        "TargetGeneExpression", "TargetGenePromoterActivityQuantile", "TargetGeneIsExpressed",
        "distance", "isSelfPromoter", "hic_contact_pl_scaled_adj",
        "ABC.Score.Numerator", "ABC.Score", "powerlaw.Score", "CellType",
    ]

    # Stream and filter
    df = stream_and_filter_abc(abc_path, gene_set, celltype_set, keep_cols, chunksize)

    # Rename
    df = df.rename(columns=RENAME_ABC)

    # Numeric conversion
    for col in ABC_NUMERIC_COLS:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    df["gene_is_expressed"] = df["gene_is_expressed"].astype(bool)
    df["is_self_promoter"] = df["is_self_promoter"].astype(bool)

    # Aggregate per (enhancer, gene, cell_type)
    df_agg = aggregate_abc_per_enhancer_gene_celltype(df)

    # Build ABC_full dict
    abc_full = aggregate_abc_to_full_dict(df_agg, present_threshold, strong_threshold)

    print(f"  Built {len(abc_full)} enhancer-gene links")
    return abc_full


def map_abc_enhancers_to_ccres(
    df_abc: pd.DataFrame,
    regulatory_elements_path: str,
) -> pd.DataFrame:
    """
    Map ABC enhancers to cCREs using center-in-interval rule.
    
    An enhancer maps to a cCRE if the enhancer's center falls within the cCRE interval.
    """
    print("Mapping ABC enhancers to cCREs...")

    # Load regulatory elements
    reg = pd.read_csv(regulatory_elements_path)
    # reg = reg.rename(columns={"ENCODE_id": "cCRE_id"})
    reg["start"] = pd.to_numeric(reg["start"], errors="coerce")
    reg["end"] = pd.to_numeric(reg["end"], errors="coerce")
    reg = reg.dropna(subset=["start", "end"])
    reg = reg.rename(columns={"start": "reg_start", "end": "reg_end"})
    reg[["reg_start", "reg_end"]] = reg[["reg_start", "reg_end"]].astype("int64")

    # Prepare ABC enhancers
    abc = df_abc.copy()
    abc["start"] = pd.to_numeric(abc["start"], errors="coerce")
    abc["end"] = pd.to_numeric(abc["end"], errors="coerce")
    abc = abc.dropna(subset=["start", "end"])
    abc = abc.rename(columns={"start": "enh_start", "end": "enh_end"})
    abc[["enh_start", "enh_end"]] = abc[["enh_start", "enh_end"]].astype("int64")
    abc["enh_center"] = ((abc["enh_start"] + abc["enh_end"]) // 2).astype("int64")

    # Map per chromosome
    mapped_chunks = []
    for chrom, reg_chr in reg.groupby("chrom"):
        abc_chr = abc[abc["chrom"] == chrom]
        if abc_chr.empty:
            continue

        reg_chr = reg_chr.sort_values("reg_start").reset_index(drop=True)
        abc_chr = abc_chr.sort_values("enh_center").reset_index(drop=True)

        merged = pd.merge_asof(
            abc_chr, reg_chr,
            left_on="enh_center", right_on="reg_start",
            direction="backward",
        )

        # Keep only centers inside cCRE
        inside = (merged["enh_center"] >= merged["reg_start"]) & (merged["enh_center"] < merged["reg_end"])
        merged = merged[inside]

        if not merged.empty:
            merged["chrom"] = chrom
            keep_cols = ["ENCODE_id", "gene_name", "ABC_full", "chrom", "enh_start", "enh_end"]
            mapped_chunks.append(merged[[c for c in keep_cols if c in merged.columns]])

    if not mapped_chunks:
        raise ValueError("No ABC enhancers could be mapped to cCREs.")

    result = pd.concat(mapped_chunks, ignore_index=True)
    result = result.drop_duplicates(
        subset=["ENCODE_id", "gene_name", "chrom", "enh_start", "enh_end"],
        keep="first",
    )

    print(f"  Mapped {len(result)} enhancer-cCRE pairs")
    return result


def collapse_abc_enhancers_per_link(df_mapped: pd.DataFrame) -> pd.DataFrame:
    """
    Collapse multiple ABC enhancers per (cCRE_id, gene_name) into a list.
    
    Returns DataFrame with columns: cCRE_id, gene_name, ABC_enhancers
    where ABC_enhancers is [{start, end, ABC_full}, ...]
    """

    def build_enhancer_list(df_sub: pd.DataFrame) -> Dict[str, List]:
        enhancers = []
        for _, r in df_sub.iterrows():
            enhancers.append({
                "start": int(r["enh_start"]),
                "end": int(r["enh_end"]),
                "ABC_full": r["ABC_full"] if "ABC_full" in df_sub.columns else {},
            })
        return {"ABC_enhancers": enhancers}

    return (
        df_mapped
        .groupby(["ENCODE_id", "gene_name"], as_index=False)
        .apply(lambda df_sub: pd.Series(build_enhancer_list(df_sub)))
    )




            





# =============================================================================
# FINAL ELEMENT TABLE
# =============================================================================

def merge_all_link_sources(
    df_screen_exp: pd.DataFrame,
    df_screen_comp: pd.DataFrame,
    df_abc_collapsed: pd.DataFrame,
) -> pd.DataFrame:
    """Merge experimental, computational, and ABC links into unified table."""
    print("Merging all link sources...")

    # Ensure string types for join columns
    for df in (df_screen_exp, df_screen_comp):
        for col in ["ENCODE_id", "gene_id", "gene_name", "gene_type"]:
            if col in df.columns:
                df[col] = df[col].astype(str)

    # Merge screen sources
    df_merged = df_screen_exp.merge(
        df_screen_comp,
        on=["ENCODE_id", "gene_id", "gene_name", "gene_type"],
        how="outer",
    )

    # Handle duplicate region columns
    if "region_comp" in df_merged.columns or "region_3D" in df_merged.columns:
        df_merged["region"] = df_merged.get("region_comp", df_merged.get("region_3D"))
        df_merged = df_merged.drop(
            columns=[c for c in ["region_3D", "region_comp"] if c in df_merged.columns]
        )

    # Attach ABC
    df_merged = df_merged.merge(df_abc_collapsed, on=["ENCODE_id", "gene_name"], how="left")

    print(f"  Merged table: {len(df_merged)} rows")
    return df_merged


def build_element_gene_links_table(
    df_links: pd.DataFrame,
    screen_exp_biosamples: List[str],
    screen_exp_assays: List[str],
    screen_comp_biosamples: List[str],
    screen_comp_assays: List[str],
) -> pd.DataFrame:
    """
    Build final element-level table with nested gene_links column.
    
    Output: DataFrame with columns [cCRE_id, gene_links]
    where gene_links is {gene_name: {gene_id, gene_type, screen_exp, screen_comp, ABC_enhancers}}
    """
    print("Building element-gene links table...")

    def build_gene_links(df_element: pd.DataFrame) -> Dict[str, Dict]:
        out = {}

        for gene_name, df_gene in df_element.groupby("gene_name", sort=False):
            base = df_gene.iloc[0].to_dict()
            base.pop("ENCODE_id", None)
            base.pop("gene_name", None)

            # Ensure screen blocks are valid
            base["screen_exp"] = ensure_screen_block(
                base.get("screen_exp"), screen_exp_biosamples, screen_exp_assays
            )
            base["screen_comp"] = ensure_screen_block(
                base.get("screen_comp"), screen_comp_biosamples, screen_comp_assays
            )

            # Handle ABC_enhancers
            if "ABC_enhancers" in df_gene.columns:
                enh_lists = [x for x in df_gene["ABC_enhancers"] if isinstance(x, (list, tuple)) and x]
                base["ABC_enhancers"] = [e for lst in enh_lists for e in lst] if enh_lists else []
            else:
                base["ABC_enhancers"] = []

            out[gene_name] = base

        return out

    gene_links_series = (
        df_links
        .groupby("ENCODE_id", sort=False)
        .apply(build_gene_links)
        .rename("gene_links")
    )

    result = gene_links_series.reset_index()
    print(f"  Built {len(result)} element records")
    return result


# =============================================================================
# MAIN EXECUTION
# =============================================================================

if __name__ == "__main__":
    # -------------------------------------------------------------------------
    # Configuration
    # -------------------------------------------------------------------------

    # Paths
    SCREEN_EXP_ZIP = r"C:\Users\stavz\Downloads\Human-Gene-Links.zip"
    SCREEN_EXP_FILE = "V4-hg38.Gene-Links.3D-Chromatin.txt"
    SCREEN_COMP_PATH = r"C:\Users\stavz\Downloads\V4-hg38.Gene-Links.Computational-Methods.txt.gz"
    REG_ELEMENTS_PATH = r"C:\Users\stavz\Desktop\masters\APM\test\regulatory_elements_matching\regulatory_element_focus.csv"
    ABC_PATH = r"C:\Users\stavz\Downloads\AllPredictions.AvgHiC.ABC0.015.minus150.ForABCPaperV3.txt"

    # Gene panel
    GENE_PANEL = [
        "PSMB8", "PSMB9", "PSMB10", "PSME1", "PSME2", "PSME4",
        "TAP1", "TAP2", "TAPBP", "TAPBPL", "CALR", "PDIA3", "PDIA6", "B2M",
        "HLA-A", "HLA-B", "HLA-C", "HLA-E", "HLA-F", "HLA-G",
        "MICA", "MICB", "ULBP1", "ULBP2", "ULBP3", "RAET1E", "RAET1G", "RAET1L",
        "PVR", "NECTIN2", "NCR3LG1", "SOCS1", "SOCS3", "NFKBIA", "TNFAIP3",
        "STAT1", "STAT3", "JAK1", "JAK2", "IRF1", "IRF2", "NLRC5",
        "CD274", "PDCD1LG2", "IFNG", "IFNGR1", "IFNGR2", "ADAM10", "ADAM17",
        "TGFB1", "IL6", "IL6R", "IL6ST", "PIAS1", "PIAS3", "PTPN2", "PTPN11",
        "EGFR", "SRC", "SMAD3", "EP300", "CREBBP", "CCL5", "CXCL9", "CXCL10", "CXCL11",
    ]

    # Biosamples / cell types
    BIOSAMPLES_SCREEN_EXP = [
        "MCF-7", "MCF_10A", "Breast Mammary Tissue",
        "breast_epithelium_tissue_female_adult_(53_years)",
        "mammary_epithelial_cell_female_adult_(19_years)",
    ]

    BIOSAMPLES_SCREEN_COMP = [
        "MCF-7", "MCF_10A", "T47D", "mammary_epithelial_cell", "breast_epithelium",
    ]

    CELLTYPES_ABC = [
        "MCF-7-ENCODE", "MCF10A-Ji2017", "MCF10A_treated_with_TAM24hr-Ji2017",
        "MDA-MB-231", "mammary_epithelial_cell-Roadmap", "breast_epithelium-ENCODE",
    ]

    # -------------------------------------------------------------------------
    # Build link tables
    # -------------------------------------------------------------------------

    # SCREEN experimental (3D chromatin)
    links_exp, assays_exp = build_screen_exp_links(
        zip_path=SCREEN_EXP_ZIP,
        inner_file=SCREEN_EXP_FILE,
        gene_list=GENE_PANEL,
        panel_biosamples=BIOSAMPLES_SCREEN_EXP,
        breast_biosamples=BIOSAMPLES_SCREEN_EXP,
        intact_pvalue_threshold=1e-6,
    )
    links_exp = collapse_screen_to_nested(links_exp, BIOSAMPLES_SCREEN_EXP, assays_exp, "screen_exp")

    # SCREEN computational
    links_comp, assays_comp = build_screen_comp_links(
        gz_path=SCREEN_COMP_PATH,
        gene_list=GENE_PANEL,
        panel_biosamples=BIOSAMPLES_SCREEN_COMP,
        breast_biosamples=BIOSAMPLES_SCREEN_COMP,
    )
    links_comp = collapse_screen_to_nested(links_comp, BIOSAMPLES_SCREEN_COMP, assays_comp, "screen_comp")

    # ABC model
    abc_raw = build_abc_enhancer_links(
        abc_path=ABC_PATH,
        gene_list=GENE_PANEL,
        celltypes=CELLTYPES_ABC,
    )
    abc_mapped = map_abc_enhancers_to_ccres(abc_raw, REG_ELEMENTS_PATH)
    abc_collapsed = collapse_abc_enhancers_per_link(abc_mapped)

    # HiCHIP


    # -------------------------------------------------------------------------
    # Merge and build final table
    # -------------------------------------------------------------------------

    links_merged = merge_all_link_sources(links_exp, links_comp, abc_collapsed)

    elements = build_element_gene_links_table(
        df_links=links_merged,
        screen_exp_biosamples=BIOSAMPLES_SCREEN_EXP,
        screen_exp_assays=assays_exp,
        screen_comp_biosamples=BIOSAMPLES_SCREEN_COMP,
        screen_comp_assays=assays_comp,
    )

    reg_df = pd.read_csv(REG_ELEMENTS_PATH)
    reg_df = reg_df.merge(elements, on="ENCODE_id", how="left")


    # -------------------------------------------------------------------------
    # Verify and output
    # -------------------------------------------------------------------------

    print(f"\n{'=' * 60}")
    print(f"Final elements table: {len(elements)} cCRE records")
    print(elements.head())

    # Verify structure
    sample = elements.iloc[0]["gene_links"]
    sample_gene = list(sample.keys())[0]
    print(f"\nSample gene_links structure for '{sample_gene}':")
    print(f"  Top-level keys: {list(sample[sample_gene].keys())}")
    print(f"  screen_exp keys: {list(sample[sample_gene]['screen_exp'].keys())}")
    print(f"  conservation_global sample: {list(sample[sample_gene]['screen_exp']['conservation_global'].keys())[:3]}...")

Building SCREEN experimental links...
  Chunk 10: kept 1597 rows
  Total rows extracted: 14841
  Assay types found: ['RNAPII-ChIAPET']
  Per-assay score quantiles:
                    q_weak  q_strong
    assay_type                      
    RNAPII-ChIAPET    56.5     87.72


C:\Users\stavz\AppData\Local\Temp\ipykernel_17880\3950273019.py:420: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  bio_series = df_bio.groupby(link_cols).apply(make_assay_dict)
C:\Users\stavz\AppData\Local\Temp\ipykernel_17880\3950273019.py:463: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return cons.groupby(link_cols).apply(to_dict).rename(column_name).reset_index()
C:\Users\stavz\AppData\Local\Temp\ipykernel_1788

  Built 4988 links
Building SCREEN computational links...
  Chunk 10: kept 1581 rows
  Total rows extracted: 12080
  Assay types found: ['ABC_(DNase_only)', 'EPIraction', 'rE2G_(DNase_only)']
  Per-assay score quantiles:
                         q_weak  q_strong
    assay_type                           
    ABC_(DNase_only)   1.000000  1.000000
    EPIraction         0.059618  0.084092
    rE2G_(DNase_only)  0.999993  0.999997


C:\Users\stavz\AppData\Local\Temp\ipykernel_17880\3950273019.py:463: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return cons.groupby(link_cols).apply(to_dict).rename(column_name).reset_index()


  Built 3947 links
Building ABC enhancer links...
  Chunk 10: kept 44 rows
  Total rows extracted: 1570
  Built 1568 enhancer-gene links
Mapping ABC enhancers to cCREs...


C:\Users\stavz\AppData\Local\Temp\ipykernel_17880\3950273019.py:379: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(build_abc_full)


  Mapped 297 enhancer-cCRE pairs
Merging all link sources...
  Merged table: 7835 rows
Building element-gene links table...


C:\Users\stavz\AppData\Local\Temp\ipykernel_17880\3950273019.py:791: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda df_sub: pd.Series(build_enhancer_list(df_sub)))


  Built 6949 element records

Final elements table: 6949 cCRE records
      ENCODE_id                                         gene_links
0  EH38E1243529  {'IRF2': {'gene_id': 'ENSG00000168310', 'gene_...
1  EH38E1355325  {'JAK1': {'gene_id': 'ENSG00000162434', 'gene_...
2  EH38E1355356  {'JAK1': {'gene_id': 'ENSG00000162434', 'gene_...
3  EH38E1355357  {'JAK1': {'gene_id': 'ENSG00000162434', 'gene_...
4  EH38E1355375  {'JAK1': {'gene_id': 'ENSG00000162434', 'gene_...

Sample gene_links structure for 'IRF2':
  Top-level keys: ['gene_id', 'gene_type', 'screen_exp', 'region', 'screen_comp', 'ABC_enhancers']
  screen_exp keys: ['per_biosample', 'conservation_global', 'conservation_breast']
  conservation_global sample: ['RNAPII-ChIAPET']...


C:\Users\stavz\AppData\Local\Temp\ipykernel_17880\3950273019.py:887: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(build_gene_links)


In [ ]:
# =============================================================================
# HiCHIP LINK TABLE BUILDERS
# =============================================================================
import os
import re
import glob
from typing import List
from collections import defaultdict

import numpy as np
import pandas as pd
import pyranges as pr


# =============================================================================
# LOOP FILE LOADING & NORMALIZATION
# =============================================================================

def load_loops_unified(path: str) -> pd.DataFrame:
    """
    Normalize different HiChIP loop formats into a standard DataFrame:
        chr1, start1, end1, chr2, start2, end2, counts, score, n_reps
    
    Supported formats:
      - LoopCatalog .bed (FitHiChIP): 4 cols with chr2:start2-end2,score
      - HMEC FitHiChIP loops.csv -> bedpe: PETs + weight
      - HiChIPper bedpe (MCF7, ZR751): weight
    """
    ext = os.path.splitext(path)[1].lower()

    if ext == ".bedpe":
        df = pd.read_csv(path, sep="\t")
    else:
        df = pd.read_csv(path, sep="\t", header=None)

    ncols = df.shape[1]

    # BEDPE with proper header
    if {"chr1", "start1", "end1", "chr2", "start2", "end2"}.issubset(df.columns):
        return _normalize_counts_score_from_bedpe(df, path)

    # 9 columns, no header
    if ncols == 9:
        df.columns = ["chr1", "start1", "end1", "chr2", "start2", "end2", "counts", "score", "n_reps"]
        return _finalize_counts_score(df, path, source="bedpe_no_header")

    # 8 columns, no header
    if ncols == 8:
        df.columns = ["chr1", "start1", "end1", "chr2", "start2", "end2", "counts", "n_reps"]
        df["score"] = np.nan
        return _finalize_counts_score(df, path, source="bedpe_no_score")

    # LoopCatalog FitHiChIP .bed – 4 columns
    if ncols == 4:
        return _parse_loopcatalog_bed(df, path)

    raise ValueError(f"Unknown loop format with {ncols} columns in {path}")


def _normalize_counts_score_from_bedpe(df: pd.DataFrame, path: str) -> pd.DataFrame:
    """Normalize BEDPE with header (HMEC/HiChIPper)."""
    coord_cols = ["chr1", "start1", "end1", "chr2", "start2", "end2"]
    missing = [c for c in coord_cols if c not in df.columns]
    if missing:
        raise ValueError(f"[bedpe] Missing coordinate columns in {path}: {missing}")

    if "PETs" in df.columns:
        counts = df["PETs"].astype(float)
    elif "weight" in df.columns:
        counts = df["weight"].astype(float)
    else:
        raise ValueError(f"[bedpe] No PETs/weight column in {path}.")

    out = df.copy()
    out["counts"] = counts
    
    if "n_reps" not in out.columns:
        out["n_reps"] = 1
    if "score" not in out.columns:
        out["score"] = np.nan

    return _finalize_counts_score(out, path, source="bedpe_with_header")


def _parse_loopcatalog_bed(df: pd.DataFrame, path: str) -> pd.DataFrame:
    """Parse LoopCatalog 4-column .bed format."""
    df.columns = ["chr1", "start1", "end1", "anchor2"]
    
    parsed = df["anchor2"].apply(lambda x: _parse_anchor2(x, path))
    df["chr2"] = parsed.str[0]
    df["start2"] = parsed.str[1].astype(int)
    df["end2"] = parsed.str[2].astype(int)
    df["score"] = parsed.str[3].astype(float)
    df["counts"] = np.nan
    df["n_reps"] = 1
    
    return _finalize_counts_score(df, path, source="loopcatalog_bed")


def _parse_anchor2(value: str, path: str) -> tuple:
    """Parse anchor2 format: chr2:start2-end2,score"""
    m = re.match(r"(chr[^:]+):(\d+)-(\d+),(.+)", str(value))
    if not m:
        raise ValueError(f"Unparseable anchor2 format in {path}: {value}")
    
    c2, s2, e2, v = m.groups()
    v = v.strip()
    score = float("inf") if v.lower() == "inf" else float(v)
    return c2, int(s2), int(e2), score


def _finalize_counts_score(df: pd.DataFrame, path: str, source: str) -> pd.DataFrame:
    """Ensure final types and column presence."""
    coord_cols = ["chr1", "start1", "end1", "chr2", "start2", "end2"]
    missing = [c for c in coord_cols if c not in df.columns]
    if missing:
        raise ValueError(f"[{source}] Missing coordinate columns in {path}: {missing}")

    out = df.copy()
    for col in ["counts", "score"]:
        if col not in out.columns:
            out[col] = np.nan
    if "n_reps" not in out.columns:
        out["n_reps"] = 1

    out["start1"] = out["start1"].astype(int)
    out["end1"] = out["end1"].astype(int)
    out["start2"] = out["start2"].astype(int)
    out["end2"] = out["end2"].astype(int)
    out["counts"] = out["counts"].astype(float)
    out["score"] = out["score"].astype(float)
    out["n_reps"] = out["n_reps"].astype(int)

    return out[["chr1", "start1", "end1", "chr2", "start2", "end2", "counts", "score", "n_reps"]]


# =============================================================================
# ANCHOR BUILDING
# =============================================================================

def build_anchors(loops: pd.DataFrame) -> pd.DataFrame:
    """Build anchor DataFrame from loops (one row per anchor A/B)."""
    if "loop_id" not in loops.columns:
        loops = loops.copy()
        loops["loop_id"] = np.arange(len(loops))

    anchor_a = loops[["chr1", "start1", "end1", "loop_id", "counts", "score", "n_reps"]].copy()
    anchor_a.rename(columns={"chr1": "chr", "start1": "start", "end1": "end"}, inplace=True)
    anchor_a["anchor"] = "A"

    anchor_b = loops[["chr2", "start2", "end2", "loop_id", "counts", "score", "n_reps"]].copy()
    anchor_b.rename(columns={"chr2": "chr", "start2": "start", "end2": "end"}, inplace=True)
    anchor_b["anchor"] = "B"

    return pd.concat([anchor_a, anchor_b], ignore_index=True)


# =============================================================================
# PYRANGES HELPERS
# =============================================================================

def _to_pyranges_coords(df: pd.DataFrame, chr_col: str = "chr") -> pd.DataFrame:
    """Rename coordinates for PyRanges compatibility."""
    rename_map = {chr_col: "Chromosome", "start": "Start", "end": "End"}
    if "chrom" in df.columns:
        rename_map["chrom"] = "Chromosome"
    return df.rename(columns=rename_map)


def _from_pyranges_coords(df: pd.DataFrame, chr_col: str = "chr") -> pd.DataFrame:
    """Rename coordinates back from PyRanges format."""
    rename_map = {"Chromosome": chr_col, "Start": "start", "End": "end"}
    return df.rename(columns=rename_map)


# =============================================================================
# FILE DISCOVERY
# =============================================================================

def find_hichip_loops_file(cell_type_hichip_dir: str, celltype: str, hichip_experiment: str) -> str:
    """Find the loops file for a given cell type and experiment."""
    candidates = [
        os.path.join(cell_type_hichip_dir, f"{celltype}_{hichip_experiment}_loops_hg38.bedpe"),
        os.path.join(cell_type_hichip_dir, f"{celltype}_{hichip_experiment}_loops_hg38.bed"),
    ]
    
    for c in candidates:
        if os.path.exists(c):
            return c

    pattern = os.path.join(cell_type_hichip_dir, "*_loops_hg38.bed*")
    matches = glob.glob(pattern)
    if matches:
        return matches[0]

    existing = os.listdir(cell_type_hichip_dir) if os.path.exists(cell_type_hichip_dir) else []
    raise FileNotFoundError(
        f"No HiChIP loops file found for {celltype} / {hichip_experiment}.\n"
        f"Looked in: {cell_type_hichip_dir}\n"
        f"Existing files: {existing}"
    )


# =============================================================================
# OVERLAP COMPUTATION
# =============================================================================

def compute_overlaps(
    loops: pd.DataFrame,
    ccre_df: pd.DataFrame,
    genes_df: pd.DataFrame
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Compute anchor overlaps with cCREs and promoters.
    
    Returns:
        tuple: (anchor_metrics, overlap_ccre, overlap_prom)
    """
    loops = loops.copy()
    if "loop_id" not in loops.columns:
        loops["loop_id"] = np.arange(len(loops))
    
    anchors = build_anchors(loops)
    
    # Separate metrics (contains NaN/inf that PyRanges can't handle)
    anchor_metrics = anchors[["loop_id", "anchor", "counts", "score", "n_reps"]].drop_duplicates()
    anchors_for_pr = anchors[["chr", "start", "end", "loop_id", "anchor"]].copy()
    
    # Prepare promoters
    promoters = genes_df[["chrom", "prom_start", "prom_end", "gene_id", "gene_name"]].copy()
    promoters.rename(columns={"prom_start": "start", "prom_end": "end"}, inplace=True)
    
    # Convert to PyRanges format
    anchors_pr = _to_pyranges_coords(anchors_for_pr, chr_col="chr")
    promoters_pr = _to_pyranges_coords(promoters, chr_col="chrom")
    ccre_pr = _to_pyranges_coords(ccre_df.copy(), chr_col="chrom")
    
    # Compute overlaps
    gr_anchors = pr.PyRanges(anchors_pr)
    gr_promoters = pr.PyRanges(promoters_pr)
    gr_ccres = pr.PyRanges(ccre_pr)
    
    overlap_prom = gr_anchors.join(gr_promoters).df
    overlap_ccre = gr_anchors.join(gr_ccres).df
    
    # Rejoin metrics
    overlap_prom = overlap_prom.merge(anchor_metrics, on=["loop_id", "anchor"], how="left")
    overlap_ccre = overlap_ccre.merge(anchor_metrics, on=["loop_id", "anchor"], how="left")
    
    return anchor_metrics, overlap_ccre, overlap_prom


# =============================================================================
# CELL LINE COLUMN INTEGRATION
# =============================================================================

def integrate_hichip_per_cell_line(
    reg_table: pd.DataFrame,
    loops_df: pd.DataFrame,
    overlap_ccre: pd.DataFrame,
    overlap_prom: pd.DataFrame,
    cell_line: str,
    hichip_key: str = "hichip",
) -> pd.DataFrame:
    """Add nested 'hichip' entry into reg_table[cell_line] dicts, per cCRE."""
    
    # Validate inputs
    _validate_overlap_columns(overlap_ccre, overlap_prom)
    
    # Ensure loop_id exists
    if "loop_id" not in loops_df.columns:
        loops_df = loops_df.copy()
        loops_df["loop_id"] = np.arange(len(loops_df))
    
    # Build lookup structures
    loop_coords = _build_loop_coords(loops_df)
    loop_to_ccres = _build_loop_to_ccres(overlap_ccre)
    loop_to_genes = _build_loop_to_genes(overlap_prom)
    
    # Build per-cCRE HiChIP data
    per_ccre = _build_per_ccre_hichip(overlap_ccre, loop_coords, loop_to_ccres, loop_to_genes)
    
    # Ensure cell_line column exists
    if cell_line not in reg_table.columns:
        reg_table[cell_line] = [{} for _ in range(len(reg_table))]
    else:
        reg_table[cell_line] = reg_table[cell_line].apply(
            lambda v: v if isinstance(v, dict) else {}
        )
    
    # Attach hichip dict per row
    def add_hichip(row):
        cell_dict = row[cell_line]
        hdata = per_ccre.get(
            str(row["cCRE_id"]),
            {"n_loops": 0, "max_counts": None, "max_score": None, "loops": []}
        )
        cell_dict[hichip_key] = hdata
        return cell_dict
    
    reg_table[cell_line] = reg_table.apply(add_hichip, axis=1)
    return reg_table


def _validate_overlap_columns(overlap_ccre: pd.DataFrame, overlap_prom: pd.DataFrame):
    """Validate required columns in overlap DataFrames."""
    required_ccre = {"cCRE_id", "loop_id", "anchor", "counts", "score", "n_reps"}
    missing = required_ccre - set(overlap_ccre.columns)
    if missing:
        raise ValueError(f"overlap_ccre missing columns: {missing}")
    
    required_prom = {"loop_id", "anchor", "gene_id", "gene_name"}
    missing = required_prom - set(overlap_prom.columns)
    if missing:
        raise ValueError(f"overlap_prom missing columns: {missing}")


def _build_loop_coords(loops_df: pd.DataFrame) -> dict:
    """Build loop_id -> anchor coordinates mapping."""
    return (
        loops_df.set_index("loop_id")
        .apply(lambda r: {
            "anchor_a": {"chr": r["chr1"], "start": int(r["start1"]), "end": int(r["end1"])},
            "anchor_b": {"chr": r["chr2"], "start": int(r["start2"]), "end": int(r["end2"])},
        }, axis=1)
        .to_dict()
    )


def _build_loop_to_ccres(overlap_ccre: pd.DataFrame) -> dict:
    """Build loop_id -> set of cCRE_ids mapping."""
    return (
        overlap_ccre.groupby("loop_id")["cCRE_id"]
        .apply(lambda s: set(s.astype(str)))
        .to_dict()
    )


def _build_loop_to_genes(overlap_prom: pd.DataFrame) -> dict:
    """Build loop_id -> {anchor -> [genes]} mapping."""
    loop_to_genes = defaultdict(lambda: defaultdict(list))
    for _, r in overlap_prom.iterrows():
        loop_to_genes[r["loop_id"]][r["anchor"]].append({
            "gene_id": str(r["gene_id"]),
            "gene_name": str(r["gene_name"]),
        })
    return loop_to_genes


def _build_per_ccre_hichip(
    overlap_ccre: pd.DataFrame,
    loop_coords: dict,
    loop_to_ccres: dict,
    loop_to_genes: dict
) -> dict:
    """Build per-cCRE HiChIP summary dictionaries."""
    per_ccre = {}
    
    for ccre_id, df_cc in overlap_ccre.groupby("cCRE_id"):
        seen_loops = set()
        loops_list = []
        max_counts = None
        max_score = None
        ccre_id_str = str(ccre_id)
        
        for _, row in df_cc.iterrows():
            lid = row["loop_id"]
            if lid in seen_loops:
                continue
            seen_loops.add(lid)
            
            cnt = float(row["counts"]) if pd.notna(row["counts"]) else None
            scr = float(row["score"]) if pd.notna(row["score"]) else None
            
            if cnt is not None:
                max_counts = cnt if max_counts is None else max(max_counts, cnt)
            if scr is not None and np.isfinite(scr):
                max_score = scr if max_score is None else max(max_score, scr)
            
            ccre_anchor = row["anchor"]
            partner_anchor = "B" if ccre_anchor == "A" else "A"
            
            partner_genes = [
                {"gene_name": g["gene_name"], "gene_id": g["gene_id"], "anchor": partner_anchor}
                for g in loop_to_genes.get(lid, {}).get(partner_anchor, [])
            ]
            
            base = {
                "loop_id": int(lid),
                "counts": cnt,
                "score": scr,
                "n_reps": int(row["n_reps"]),
                "partner_genes": partner_genes,
                "partner_cCREs": sorted(loop_to_ccres.get(lid, set()) - {ccre_id_str}),
            }
            
            coords = loop_coords.get(lid)
            if coords:
                base.update(coords)
            
            loops_list.append(base)
        
        if loops_list:
            per_ccre[ccre_id_str] = {
                "n_loops": len(loops_list),
                "max_counts": max_counts,
                "max_score": max_score,
                "loops": loops_list,
            }
    
    return per_ccre


# =============================================================================
# GENE LINKS INTEGRATION
# =============================================================================

def build_ccre_gene_hichip_map(
    overlap_ccre: pd.DataFrame,
    overlap_prom: pd.DataFrame
) -> dict:
    """Build cCRE -> gene -> [loops] mapping for opposite-anchor pairs."""
    cc = overlap_ccre.rename(columns={"anchor": "anchor_ccre"})
    cp = overlap_prom.rename(columns={"anchor": "anchor_prom"})
    
    merged = cc.merge(cp, on="loop_id", suffixes=("_ccre", "_prom"))
    merged_ep = merged[merged["anchor_ccre"] != merged["anchor_prom"]]
    
    cg = defaultdict(lambda: defaultdict(list))
    for _, r in merged_ep.iterrows():
        cg[r["cCRE_id"]][r["gene_name"]].append({
            "loop_id": r["loop_id"],
            "counts": float(r["counts_ccre"]) if pd.notna(r["counts_ccre"]) else None,
            "score": float(r["score_ccre"]) if pd.notna(r["score_ccre"]) else None,
            "n_reps": r["n_reps_ccre"],
            "anchor_ccre": r["anchor_ccre"],
            "anchor_prom": r["anchor_prom"],
        })
    
    return cg



def add_hichip_to_gene_links(
    gene_links: dict,
    ccre_id: str,
    cell_line: str,
    ccre_gene_map: dict,
) -> dict:
    """
    Add HiChIP data to gene_links dict.

    - Each gene always keeps a 'hichip' dict (may be empty).
    - Inside 'hichip', a given cell_line appears ONLY if there are loops.
    """
    # All loops for this cCRE, organized as {gene_name: [loop_dict, ...]}
    per_gene_loops = ccre_gene_map.get(ccre_id, {})

    for gene_name, gdict in gene_links.items():
        # Ensure per-gene "hichip" container exists
        hichip = gdict.setdefault("hichip", {})

        loops_for_pair = per_gene_loops.get(gene_name, [])

        # If no loops for this (cCRE, gene, cell_line), do NOT create an entry
        if not loops_for_pair:
            # Important: do *not* touch hichip[cell_line] here
            # so no empty dicts get created for this biosample.
            continue

        counts_vals = [
            l.get("counts")
            for l in loops_for_pair
            if l.get("counts") is not None
        ]
        score_vals = [
            l.get("score")
            for l in loops_for_pair
            if l.get("score") is not None and np.isfinite(l.get("score"))
        ]

        hichip[cell_line] = {
            "n_loops": len(loops_for_pair),
            "max_counts": max(counts_vals) if counts_vals else None,
            "max_score": max(score_vals) if score_vals else None,
            "loops": loops_for_pair,
        }

    return gene_links



# =============================================================================
# MAIN PROCESSING FUNCTIONS
# =============================================================================

def handle_cell_type_hichip(
    cell_type_hichip_dir: str,
    hichip_experiment: str,
    celltype: str,
    ccre_df: pd.DataFrame,
    genes_df: pd.DataFrame
) -> pd.DataFrame:
    """Process HiChIP data for a single cell type."""
    loops_path = find_hichip_loops_file(cell_type_hichip_dir, celltype, hichip_experiment)
    print(f"[HiChIP] Using loops file for {celltype}: {loops_path}")
    
    loops = load_loops_unified(loops_path)
    loops["loop_id"] = np.arange(len(loops))
    
    # Compute overlaps
    _, overlap_ccre, overlap_prom = compute_overlaps(loops, ccre_df, genes_df)
    
    # Integrate into cell line column
    ccre_df = integrate_hichip_per_cell_line(ccre_df, loops, overlap_ccre, overlap_prom, celltype)
    
    # Build cCRE-gene HiChIP map and update gene_links
    ccre_gene_map = build_ccre_gene_hichip_map(overlap_ccre, overlap_prom)
    
    ccre_df["gene_links"] = ccre_df.apply(
        lambda row: add_hichip_to_gene_links(
            row["gene_links"], row["cCRE_id"], celltype, ccre_gene_map
        ) if pd.notna(row["gene_links"]) and isinstance(row["gene_links"], dict) else row["gene_links"],
        axis=1
    )
    
    return ccre_df


def merge_all_hichips(
    hichip_dir: str,
    hichip_experiment: str,
    hichip_panel: List[str],
    ccre_df: pd.DataFrame,
    genes_df: pd.DataFrame
) -> pd.DataFrame:
    """Process HiChIP data for all cell types in panel."""
    for celltype in hichip_panel:
        cell_type_hichip_dir = os.path.join(hichip_dir, celltype)
        if os.path.exists(cell_type_hichip_dir):
            ccre_df = handle_cell_type_hichip(
                cell_type_hichip_dir, hichip_experiment, celltype, ccre_df, genes_df
            )
        else:
            print(f"[HiChIP] Skipping {celltype}: directory not found")
    
    return ccre_df


# =============================================================================
# MAIN EXECUTION
# =============================================================================

if __name__ == "__main__":
    # Configuration
    HICHIP_PANEL = ["B80T5", "HMEC", "K5plusK19plus", "MCF7", "MDA-MB-231", "T47D", "ZR751"]
    HICHIP_DIR = r'C:\Users\stavz\Desktop\masters\APM\data\HiCHIP'
    HICHIP_EXPERIMENT = "H3K27ac"
    reg_df.rename(columns={"elem_id": "cCRE_id"}, inplace=True)
    
    # Run (assumes reg_df and genes_df are defined)
    cCRE_df = merge_all_hichips(HICHIP_DIR, HICHIP_EXPERIMENT, HICHIP_PANEL, reg_df, genes_df)

[HiChIP] Using loops file for B80T5: C:\Users\stavz\Desktop\masters\APM\data\HiCHIP\B80T5\B80T5_H3K27ac_loops_hg38.bed
[HiChIP] Using loops file for HMEC: C:\Users\stavz\Desktop\masters\APM\data\HiCHIP\HMEC\HMEC_H3K27ac_loops_hg38.bedpe
[HiChIP] Using loops file for K5plusK19plus: C:\Users\stavz\Desktop\masters\APM\data\HiCHIP\K5plusK19plus\K5plusK19plus_H3K27ac_loops_hg38.bed
[HiChIP] Using loops file for MCF7: C:\Users\stavz\Desktop\masters\APM\data\HiCHIP\MCF7\MCF7_H3K27ac_loops_hg38.bedpe
[HiChIP] Using loops file for MDA-MB-231: C:\Users\stavz\Desktop\masters\APM\data\HiCHIP\MDA-MB-231\MDA-MB-231_H3K27ac_loops_hg38.bed
[HiChIP] Using loops file for T47D: C:\Users\stavz\Desktop\masters\APM\data\HiCHIP\T47D\T47D_H3K27ac_loops_hg38.bed
[HiChIP] Using loops file for ZR751: C:\Users\stavz\Desktop\masters\APM\data\HiCHIP\ZR751\ZR751_H3K27ac_loops_hg38.bedpe


In [157]:
gene_links_df = cCRE_df[cCRE_df["gene_links"].notna()]
gene_links_df["gene_links"].to_csv("test/gene_links_df_adhoc.csv", index=False)


In [ ]:
abc_raw.to_csv("data/regulatory_elements_data/ABC_output/abc_raw.csv", index=False)
elements.to_csv("data/regulatory_elements_data/cCRE_gene_links.csv", index=False)
links_comp.to_csv("data/regulatory_elements_data/SCREEN_comp/screen_comp_links.csv", index=False)
links_exp.to_csv("data/regulatory_elements_data/SCREEN_exp/screen_exp_links.csv", index=False)

In [158]:
reg_df = pd.read_csv(r'C:\Users\stavz\Desktop\masters\APM\test\regulatory_elements_matching\regulatory_element_focus.csv')
reg_df.rename(columns={"elem_id": "cCRE_id"}, inplace=True)
reg_df.to_csv(r'C:\Users\stavz\Desktop\masters\APM\test\regulatory_elements_matching\regulatory_element_focus.csv')

In [2]:
import pandas as pd
mirna = pd.read_csv(r"data/miRNA/Predicted_Targets_Context_Scores.default_predictions.txt", sep = "\t")
mirna

,Gene ID,Gene Symbol,Transcript ID,Gene Tax ID,miRNA,Site Type,UTR_start,UTR end,context++ score,context++ score percentile,weighted context++ score,weighted context++ score percentile,Predicted relative KD
0,ENSG00000121410.7,A1BG,ENST00000263100.3,9544,mml-miR-23a-3p,3,142,149,-0.428,97,-0.388,97,NaN
1,ENSG00000121410.7,A1BG,ENST00000263100.3,9544,mml-miR-23b-3p,3,142,149,-0.428,97,-0.388,97,NaN
2,ENSG00000121410.7,A1BG,ENST00000263100.3,9598,ptr-miR-23a,3,143,150,-0.419,97,-0.419,98,NaN
3,ENSG00000121410.7,A1BG,ENST00000263100.3,9598,ptr-miR-23b,3,143,150,-0.419,97,-0.419,98,NaN
4,ENSG00000121410.7,A1BG,ENST00000263100.3,9598,ptr-miR-23c,3,143,150,-0.419,97,-0.419,98,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1397973,ENSG00000036549.8,ZZZ3,ENST00000370801.3,10116,rno-miR-128-3p,1,1029,1035,-0.084,72,-0.028,63,NaN
1397974,ENSG00000036549.8,ZZZ3,ENST00000370801.3,10116,rno-miR-342-3p,1,1031,1037,-0.010,39,-0.003,48,NaN
1397975,ENSG00000036549.8,ZZZ3,ENST00000370801.3,10116,rno-miR-369-3p,1,1065,1071,-0.010,49,-0.003,57,NaN
1397976,ENSG00000036549.8,ZZZ3,ENST00000370801.3,10116,rno-miR-374-5p,2,1065,1071,-0.020,41,-0.007,46,NaN


In [3]:
mirna

,Gene ID,Gene Symbol,Transcript ID,Gene Tax ID,miRNA,Site Type,UTR_start,UTR end,context++ score,context++ score percentile,weighted context++ score,weighted context++ score percentile,Predicted relative KD
0,ENSG00000121410.7,A1BG,ENST00000263100.3,9544,mml-miR-23a-3p,3,142,149,-0.428,97,-0.388,97,NaN
1,ENSG00000121410.7,A1BG,ENST00000263100.3,9544,mml-miR-23b-3p,3,142,149,-0.428,97,-0.388,97,NaN
2,ENSG00000121410.7,A1BG,ENST00000263100.3,9598,ptr-miR-23a,3,143,150,-0.419,97,-0.419,98,NaN
3,ENSG00000121410.7,A1BG,ENST00000263100.3,9598,ptr-miR-23b,3,143,150,-0.419,97,-0.419,98,NaN
4,ENSG00000121410.7,A1BG,ENST00000263100.3,9598,ptr-miR-23c,3,143,150,-0.419,97,-0.419,98,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1397973,ENSG00000036549.8,ZZZ3,ENST00000370801.3,10116,rno-miR-128-3p,1,1029,1035,-0.084,72,-0.028,63,NaN
1397974,ENSG00000036549.8,ZZZ3,ENST00000370801.3,10116,rno-miR-342-3p,1,1031,1037,-0.010,39,-0.003,48,NaN
1397975,ENSG00000036549.8,ZZZ3,ENST00000370801.3,10116,rno-miR-369-3p,1,1065,1071,-0.010,49,-0.003,57,NaN
1397976,ENSG00000036549.8,ZZZ3,ENST00000370801.3,10116,rno-miR-374-5p,2,1065,1071,-0.020,41,-0.007,46,NaN


In [ ]:
# =============================================================================
# ENHANCER-GENE LINK TABLE BUILDERS
# =============================================================================
# Integrates:
#   - ENCODE SCREEN experimental (3D chromatin)
#   - ENCODE SCREEN computational
#   - ABC model predictions
#   - HiChIP loop data
# =============================================================================

import os
import re
import glob
import zipfile
from typing import Iterable, Optional, Dict, Any, List, Set, Tuple
from collections import defaultdict

import numpy as np
import pandas as pd
import pyranges as pr


# =============================================================================
# SCHEMA HELPERS
# =============================================================================

def empty_assay_entry() -> Dict[str, Any]:
    return {"score": None, "p_value": None, "strength": "none"}


def empty_biosample_assays(assay_types: List[str]) -> Dict[str, Dict[str, Any]]:
    return {assay: empty_assay_entry() for assay in assay_types}


def empty_conservation_entry() -> Dict[str, int]:
    return {"n_biosamples": 0, "n_strong": 0, "n_weak": 0}


def empty_conservation_block(assay_types: List[str]) -> Dict[str, Dict[str, int]]:
    return {assay: empty_conservation_entry() for assay in assay_types}


def empty_screen_block(biosamples: List[str], assay_types: List[str]) -> Dict[str, Any]:
    return {
        "per_biosample": {bio: empty_biosample_assays(assay_types) for bio in biosamples},
        "conservation_global": empty_conservation_block(assay_types),
        "conservation_breast": empty_conservation_block(assay_types),
    }


def ensure_screen_block(
    val: Any,
    biosamples: List[str],
    assay_types: List[str],
) -> Dict[str, Any]:
    """Coerce any value into a valid screen block, filling missing keys."""
    if not isinstance(val, dict):
        return empty_screen_block(biosamples, assay_types)

    val.setdefault("per_biosample", {})
    val.setdefault("conservation_global", {})
    val.setdefault("conservation_breast", {})

    for bio in biosamples:
        if bio not in val["per_biosample"] or not isinstance(val["per_biosample"][bio], dict):
            val["per_biosample"][bio] = empty_biosample_assays(assay_types)
        else:
            for assay in assay_types:
                val["per_biosample"][bio].setdefault(assay, empty_assay_entry())

    for cons_key in ["conservation_global", "conservation_breast"]:
        if not isinstance(val[cons_key], dict):
            val[cons_key] = empty_conservation_block(assay_types)
        else:
            for assay in assay_types:
                val[cons_key].setdefault(assay, empty_conservation_entry())

    return val


def empty_abc_celltype_entry() -> Dict[str, Any]:
    return {
        "ABC_score": None,
        "ABC_num": None,
        "activity": None,
        "distance": None,
        "element_class": None,
        "is_self_promoter": False,
        "hic_pl_scaled": None,
        "powerlaw_score": None,
        "gene_expr": None,
        "promoter_activity_q": None,
        "gene_is_expressed": False,
        "rank_within_gene": None,
        "is_present": False,
        "is_strong": False,
    }


def empty_abc_block(celltypes: List[str]) -> Dict[str, Dict[str, Any]]:
    return {ct: empty_abc_celltype_entry() for ct in celltypes}


# =============================================================================
# COLUMN DEFINITIONS & RENAMING
# =============================================================================

COLS_SCREEN_EXP = [
    "cCRE ID", "Gene ID", "Common Gene Name", "Gene Type",
    "Assay Type", "Experiment ID", "Biosample", "Score", "P-value",
]

COLS_SCREEN_COMP = [
    "cCRE ID", "Gene ID", "Common Gene Name", "Gene Type",
    "Assay Type", "Region", "Experiment ID", "Biosample", "Score",
]

COLS_ABC_RAW = [
    "chr", "start", "end", "name", "class", "activity_base",
    "TargetGene", "TargetGeneTSS", "TargetGeneExpression",
    "TargetGenePromoterActivityQuantile", "TargetGeneIsExpressed",
    "distance", "isSelfPromoter", "hic_contact", "powerlaw_contact",
    "powerlaw_contact_reference", "hic_contact_pl_scaled", "hic_pseudocount",
    "hic_contact_pl_scaled_adj", "ABC.Score.Numerator", "ABC.Score",
    "powerlaw.Score.Numerator", "powerlaw.Score", "CellType",
]

RENAME_SCREEN = {
    "cCRE ID": "ENCODE_id",
    "Gene ID": "gene_id",
    "Common Gene Name": "gene_name",
    "Gene Type": "gene_type",
    "Assay Type": "assay_type",
    "Experiment ID": "experiment_id",
    "Biosample": "biosample",
    "Score": "score",
    "P-value": "p_value",
    "Region": "region",
}

RENAME_ABC = {
    "name": "enh_id",
    "chr": "chrom",
    "class": "element_class",
    "activity_base": "activity",
    "TargetGene": "gene_name",
    "TargetGeneExpression": "gene_expr",
    "TargetGenePromoterActivityQuantile": "promoter_activity_q",
    "TargetGeneIsExpressed": "gene_is_expressed",
    "isSelfPromoter": "is_self_promoter",
    "hic_contact_pl_scaled_adj": "hic_pl_scaled",
    "ABC.Score.Numerator": "ABC_num",
    "ABC.Score": "ABC_score",
    "powerlaw.Score": "powerlaw_score",
    "CellType": "cell_type",
}

LINK_COLS = ["ENCODE_id", "gene_id", "gene_name"]

ABC_NUMERIC_COLS = [
    "start", "end", "activity", "gene_expr", "promoter_activity_q",
    "distance", "hic_pl_scaled", "ABC_num", "ABC_score", "powerlaw_score"
]


# =============================================================================
# STREAMING & FILTERING
# =============================================================================

def stream_and_filter_genes(
    file_handle,
    gene_set: Set[str],
    columns: List[str],
    chunksize: int,
    sep: str = "\t",
    compression: Optional[str] = None,
    add_dummy_pvalue: bool = False,
) -> pd.DataFrame:
    """Stream a tabular file and retain only rows matching the gene set."""
    chunks = []
    reader = pd.read_csv(
        file_handle,
        sep=sep,
        header=None,
        names=columns,
        chunksize=chunksize,
        low_memory=False,
        compression=compression,
    )

    for i, chunk in enumerate(reader):
        chunk["Common Gene Name"] = chunk["Common Gene Name"].astype(str).str.strip()
        chunk["Score"] = pd.to_numeric(chunk["Score"], errors="coerce")

        if i == 10:
            break

        if add_dummy_pvalue:
            chunk["P-value"] = np.nan

        mask = chunk["Common Gene Name"].isin(gene_set)
        if mask.any():
            chunks.append(chunk.loc[mask])

        if (i + 1) % 10 == 0:
            print(f"  Chunk {i + 1}: kept {mask.sum()} rows")

    if not chunks:
        raise ValueError("No rows found for selected genes.")

    df = pd.concat(chunks, ignore_index=True)
    print(f"  Total rows extracted: {len(df)}")
    return df


def stream_and_filter_abc(
    file_path: str,
    gene_set: Set[str],
    celltype_set: Optional[Set[str]],
    keep_cols: List[str],
    chunksize: int,
) -> pd.DataFrame:
    """Stream ABC file and retain rows matching genes and cell types."""
    chunks = []
    reader = pd.read_csv(
        file_path, sep="\t", header=0, names=COLS_ABC_RAW,
        chunksize=chunksize, low_memory=False, compression="infer",
    )

    for i, chunk in enumerate(reader):
        chunk["TargetGene"] = chunk["TargetGene"].astype(str).str.strip()
        chunk["CellType"] = chunk["CellType"].astype(str).str.strip()
        chunk.rename(columns={"chr": "chrom"}, inplace=True)

        mask = chunk["TargetGene"].isin(gene_set)
        if celltype_set:
            mask &= chunk["CellType"].isin(celltype_set)

        if mask.any():
            chunks.append(chunk.loc[mask, keep_cols].copy())

        if (i + 1) % 10 == 0:
            print(f"  Chunk {i + 1}: kept {mask.sum()} rows")

    if not chunks:
        raise ValueError("No ABC rows found for selected genes/cell types.")

    df = pd.concat(chunks, ignore_index=True)
    print(f"  Total rows extracted: {len(df)}")
    return df


# =============================================================================
# STRENGTH CLASSIFICATION
# =============================================================================

def classify_strength(is_strong: bool, is_weak: bool) -> str:
    if is_strong:
        return "strong"
    return "weak" if is_weak else "none"


def apply_strength_classification(
    df: pd.DataFrame,
    weak_quantile: float,
    strong_quantile: float,
    intact_pvalue_threshold: Optional[float] = None,
) -> pd.DataFrame:
    """Compute per-assay quantiles and assign strength labels."""
    quantiles = (
        df.groupby("assay_type")["score"]
        .quantile([weak_quantile, strong_quantile])
        .unstack()
        .rename(columns={weak_quantile: "q_weak", strong_quantile: "q_strong"})
    )
    print("  Per-assay score quantiles:")
    for line in quantiles.to_string().split('\n'):
        print(f"    {line}")

    df = df.merge(quantiles, left_on="assay_type", right_index=True, how="left")

    df["is_strong"] = df["score"] >= df["q_strong"]
    df["is_weak"] = (df["score"] >= df["q_weak"]) & ~df["is_strong"]

    if intact_pvalue_threshold is not None:
        mask = df["assay_type"] == "Intact-HiC"
        df.loc[mask, "is_strong"] &= df.loc[mask, "p_value"] <= intact_pvalue_threshold

    df["strength"] = [classify_strength(s, w) for s, w in zip(df["is_strong"], df["is_weak"])]
    return df


# =============================================================================
# AGGREGATION HELPERS
# =============================================================================

def aggregate_screen_per_link_biosample_assay(
    df: pd.DataFrame,
    link_cols: List[str],
    include_region: bool = False,
) -> pd.DataFrame:
    """Aggregate SCREEN data to one row per (link, biosample, assay_type)."""
    group_cols = link_cols + ["biosample", "assay_type"]

    agg_spec = {
        "score": ("score", "max"),          # max score per assay (Permissive)
        "p_value": ("p_value", "min"),      # min p-value per assay (Permissive)
        "any_strong": ("is_strong", "max"), 
        "any_weak": ("is_weak", "max"),    
        "gene_type": ("gene_type", "first"), 
    }
    if include_region and "region" in df.columns:
        agg_spec["region"] = ("region", "first")

    result = df.groupby(group_cols).agg(**agg_spec).reset_index()
    result["strength"] = [
        classify_strength(s, w) for s, w in zip(result["any_strong"], result["any_weak"]) # classifying limk in 
    ]
    return result


def aggregate_abc_per_enhancer_gene_celltype(df: pd.DataFrame) -> pd.DataFrame:
    """Aggregate ABC data to one row per (enhancer, gene, cell_type)."""
    group_cols = ["chrom", "start", "end", "enh_id", "gene_name", "cell_type"]

    return df.groupby(group_cols).agg(
        element_class=("element_class", "first"),
        activity=("activity", "max"),
        distance=("distance", "median"),
        is_self_promoter=("is_self_promoter", "max"),
        hic_pl_scaled=("hic_pl_scaled", "max"),
        ABC_num=("ABC_num", "max"),
        ABC_score=("ABC_score", "max"),
        powerlaw_score=("powerlaw_score", "max"),
        gene_expr=("gene_expr", "first"),
        promoter_activity_q=("promoter_activity_q", "first"),
        gene_is_expressed=("gene_is_expressed", "max"),
    ).reset_index()


def aggregate_abc_to_full_dict(
    df_agg: pd.DataFrame,
    present_threshold: float,
    strong_threshold: float,
) -> pd.DataFrame:
    """Add presence/strength flags and collapse to ABC_full dict per (enhancer, gene)."""
    df_agg = df_agg.copy()
    df_agg["is_present"] = df_agg["ABC_score"] >= present_threshold   ### To think about about discarding promoters ###
    df_agg["is_strong"] = (df_agg["ABC_score"] >= strong_threshold) & (df_agg["element_class"] != "promoter")

    max_score = df_agg.groupby(["gene_name", "cell_type"])["ABC_score"].transform("max")
    df_agg["rank_within_gene"] = np.where(max_score > 0, df_agg["ABC_score"] / max_score, 0.0)

    def build_abc_full(df_sub: pd.DataFrame) -> Dict[str, Dict[str, Any]]:
        out = {}
        for _, r in df_sub.iterrows():             ### Maybe to think about all the None values ###
            out[r["cell_type"]] = {
                "ABC_score": float(r["ABC_score"]) if pd.notna(r["ABC_score"]) else None,
                "ABC_num": float(r["ABC_num"]) if pd.notna(r["ABC_num"]) else None,
                "activity": float(r["activity"]) if pd.notna(r["activity"]) else None,
                "distance": int(r["distance"]) if pd.notna(r["distance"]) else None,
                "element_class": r["element_class"],
                "is_self_promoter": bool(r["is_self_promoter"]),
                "hic_pl_scaled": float(r["hic_pl_scaled"]) if pd.notna(r["hic_pl_scaled"]) else None,
                "powerlaw_score": float(r["powerlaw_score"]) if pd.notna(r["powerlaw_score"]) else None,
                "gene_expr": float(r["gene_expr"]) if pd.notna(r["gene_expr"]) else None,
                "promoter_activity_q": float(r["promoter_activity_q"]) if pd.notna(r["promoter_activity_q"]) else None,
                "gene_is_expressed": bool(r["gene_is_expressed"]),
                "rank_within_gene": float(r["rank_within_gene"]) if pd.notna(r["rank_within_gene"]) else None,
                "is_present": bool(r["is_present"]),
                "is_strong": bool(r["is_strong"]),
            }
        return out

    return (
        df_agg.groupby(["chrom", "start", "end", "enh_id", "gene_name"], sort=False)
        .apply(build_abc_full)
        .rename("ABC_full")
        .reset_index()
    )


# =============================================================================
# DICT BUILDERS FOR SCREEN DATA
# =============================================================================

def build_biosample_columns(
    df_aggregated: pd.DataFrame,
    df_links: pd.DataFrame,
    link_cols: List[str],
    biosamples: List[str],
    assay_types: List[str],
) -> pd.DataFrame:
    """Add one column per biosample with nested {assay: {score, p_value, strength}}."""

    def make_assay_dict(df_sub: pd.DataFrame) -> Dict[str, Dict[str, Any]]:
        rows_by_assay = {r["assay_type"]: r for _, r in df_sub.iterrows()}
        out = {}
        for assay in assay_types:
            if assay in rows_by_assay:
                r = rows_by_assay[assay]
                out[assay] = {
                    "score": float(r["score"]) if pd.notna(r["score"]) else None,
                    "p_value": float(r["p_value"]) if pd.notna(r["p_value"]) else None,
                    "strength": r["strength"],
                }
            else:
                out[assay] = empty_assay_entry()
        return out

    default = empty_biosample_assays(assay_types)

    for bio in biosamples:
        df_bio = df_aggregated[df_aggregated["biosample"] == bio]
        if df_bio.empty:
            df_links[bio] = [default.copy() for _ in range(len(df_links))]
        else:
            bio_series = df_bio.groupby(link_cols).apply(make_assay_dict)
            df_links = df_links.merge(bio_series.rename(bio).reset_index(), on=link_cols, how="left")
            df_links[bio] = df_links[bio].apply(lambda x: x if isinstance(x, dict) else default.copy())

    return df_links


def build_conservation_column(
    df_aggregated: pd.DataFrame,
    link_cols: List[str],
    biosample_set: Set[str],
    assay_types: List[str],
    column_name: str,
) -> pd.DataFrame:
    """Build conservation summary: {assay_type: {n_biosamples, n_strong, n_weak}}."""
    filtered = df_aggregated[df_aggregated["biosample"].isin(biosample_set)]

    if filtered.empty:
        return None

    cons = (
        filtered.groupby(link_cols + ["assay_type"])
        .agg(
            n_biosamples=("biosample", "nunique"),
            n_strong=("any_strong", "sum"),
            n_weak=("any_weak", "sum"),
        )
        .reset_index()
    )

    def to_dict(df_sub: pd.DataFrame) -> Dict[str, Dict[str, int]]:
        result = {r["assay_type"]: {
            "n_biosamples": int(r["n_biosamples"]),
            "n_strong": int(r["n_strong"]),
            "n_weak": int(r["n_weak"]),
        } for _, r in df_sub.iterrows()}
        
        for assay in assay_types:
            if assay not in result:
                result[assay] = empty_conservation_entry()
        return result

    return cons.groupby(link_cols).apply(to_dict).rename(column_name).reset_index()


def attach_conservation_columns(
    df_links: pd.DataFrame,
    df_aggregated: pd.DataFrame,
    link_cols: List[str],
    assay_types: List[str],
    global_biosamples: Optional[Set[str]],
    breast_biosamples: Set[str],
) -> pd.DataFrame:
    """Attach conservation_global and conservation_breast columns."""
    default_cons = empty_conservation_block(assay_types)

    global_subset = df_aggregated if global_biosamples is None else \
        df_aggregated[df_aggregated["biosample"].isin(global_biosamples)]
    
    cons_global = build_conservation_column(
        global_subset, link_cols, set(global_subset["biosample"]), assay_types, "conservation_global"
    )
    if cons_global is not None:
        df_links = df_links.merge(cons_global, on=link_cols, how="left")
    
    df_links["conservation_global"] = df_links.get("conservation_global", pd.Series([None] * len(df_links))).apply(
        lambda x: x if isinstance(x, dict) else default_cons.copy()
    )

    cons_breast = build_conservation_column(
        df_aggregated, link_cols, breast_biosamples, assay_types, "conservation_breast"
    )
    if cons_breast is not None:
        df_links = df_links.merge(cons_breast, on=link_cols, how="left")
    
    df_links["conservation_breast"] = df_links.get("conservation_breast", pd.Series([None] * len(df_links))).apply(
        lambda x: x if isinstance(x, dict) else default_cons.copy()
    )

    return df_links


# =============================================================================
# SCREEN LINK TABLE BUILDERS
# =============================================================================

def build_screen_exp_links(
    zip_path: str,
    inner_file: str,
    gene_list: Iterable[str],
    panel_biosamples: List[str],
    breast_biosamples: Optional[List[str]] = None,
    global_biosamples: Optional[List[str]] = None,
    weak_quantile: float = 0.90,
    strong_quantile: float = 0.95,
    intact_pvalue_threshold: Optional[float] = None,
    chunksize: int = 300_000,
) -> Tuple[pd.DataFrame, List[str]]:
    """Build enhancer-gene links from ENCODE SCREEN 3D-Chromatin (experimental) data."""
    print("Building SCREEN experimental links...")

    gene_set = set(gene_list)
    breast_set = set(breast_biosamples or panel_biosamples)
    global_set = set(global_biosamples) if global_biosamples else None

    with zipfile.ZipFile(zip_path) as z:
        with z.open(inner_file) as f:
            df = stream_and_filter_genes(f, gene_set, COLS_SCREEN_EXP, chunksize)

    df = df.rename(columns=RENAME_SCREEN)
    df["p_value"] = pd.to_numeric(df["p_value"], errors="coerce") ### maybe check this conversion to numeric

    assay_types = sorted(df["assay_type"].dropna().unique())
    print(f"  Assay types found: {assay_types}")

    df = apply_strength_classification(df, weak_quantile, strong_quantile, intact_pvalue_threshold)
    df_agg = aggregate_screen_per_link_biosample_assay(df, LINK_COLS)
    links = df_agg[LINK_COLS + ["gene_type"]].drop_duplicates().reset_index(drop=True)
    links = build_biosample_columns(df_agg, links, LINK_COLS, panel_biosamples, assay_types)
    links = attach_conservation_columns(links, df_agg, LINK_COLS, assay_types, global_set, breast_set)

    print(f"  Built {len(links)} links")
    return links, assay_types


def build_screen_comp_links(
    gz_path: str,
    gene_list: Iterable[str],
    panel_biosamples: List[str],
    breast_biosamples: Optional[List[str]] = None,
    global_biosamples: Optional[List[str]] = None,
    weak_quantile: float = 0.90,
    strong_quantile: float = 0.95,
    chunksize: int = 300_000,
) -> Tuple[pd.DataFrame, List[str]]:
    """Build enhancer-gene links from ENCODE SCREEN Computational-Methods data."""
    print("Building SCREEN computational links...")

    gene_set = set(gene_list)
    breast_set = set(breast_biosamples or panel_biosamples)
    global_set = set(global_biosamples) if global_biosamples else None

    df = stream_and_filter_genes(
        gz_path, gene_set, COLS_SCREEN_COMP, chunksize,
        compression="gzip", add_dummy_pvalue=True
    )

    df = df.rename(columns=RENAME_SCREEN)
    df["p_value"] = pd.to_numeric(df["p_value"], errors="coerce")

    assay_types = sorted(df["assay_type"].dropna().unique())
    print(f"  Assay types found: {assay_types}")

    # critical step - asigning strength. thresholds are currently evidence-type specific.
    df = apply_strength_classification(df, weak_quantile, strong_quantile) 
    #**********************************************************************************

      
    df_agg = aggregate_screen_per_link_biosample_assay(df, LINK_COLS, include_region=True)
    links = df_agg[LINK_COLS + ["gene_type"]].drop_duplicates().reset_index(drop=True)

    region_series = df_agg.groupby(LINK_COLS)["region"].first().reset_index()
    links = links.merge(region_series, on=LINK_COLS, how="left")

    links = build_biosample_columns(df_agg, links, LINK_COLS, panel_biosamples, assay_types)
    links = attach_conservation_columns(links, df_agg, LINK_COLS, assay_types, global_set, breast_set)

    print(f"  Built {len(links)} links")
    return links, assay_types


def collapse_screen_to_nested(
    df_links: pd.DataFrame,
    biosamples: List[str],
    assay_types: List[str],
    target_column: str,
) -> pd.DataFrame:
    """Collapse biosample columns into a single nested dict column."""

    def row_to_nested(row) -> Dict[str, Any]:
        per_bio = {}
        for bio in biosamples:
            if bio in row.index and isinstance(row[bio], dict):
                per_bio[bio] = row[bio]
            else:
                per_bio[bio] = empty_biosample_assays(assay_types)

        cons_global = row.get("conservation_global")
        cons_breast = row.get("conservation_breast")

        return {
            "per_biosample": per_bio,
            "conservation_global": cons_global if isinstance(cons_global, dict) else empty_conservation_block(assay_types),
            "conservation_breast": cons_breast if isinstance(cons_breast, dict) else empty_conservation_block(assay_types),
        }

    df_links[target_column] = df_links.apply(row_to_nested, axis=1)

    drop_cols = [c for c in biosamples + ["conservation_global", "conservation_breast"] if c in df_links.columns]
    return df_links.drop(columns=drop_cols)


# =============================================================================
# ABC LINK TABLE BUILDERS
# =============================================================================

def build_abc_enhancer_links(
    abc_path: str,
    gene_list: Iterable[str],
    celltypes: Optional[List[str]] = None,
    present_threshold: float = 0.015,
    strong_threshold: float = 0.05,
    chunksize: int = 500_000,
) -> pd.DataFrame:
    """Build enhancer-gene links from ABC model predictions."""
    print("Building ABC enhancer links...")

    gene_set = set(gene_list)
    celltype_set = set(celltypes) if celltypes else None

    keep_cols = [
        "chrom", "start", "end", "name", "class", "activity_base", "TargetGene",
        "TargetGeneExpression", "TargetGenePromoterActivityQuantile", "TargetGeneIsExpressed",
        "distance", "isSelfPromoter", "hic_contact_pl_scaled_adj",
        "ABC.Score.Numerator", "ABC.Score", "powerlaw.Score", "CellType",
    ]

    df = stream_and_filter_abc(abc_path, gene_set, celltype_set, keep_cols, chunksize)
    df = df.rename(columns=RENAME_ABC)

    for col in ABC_NUMERIC_COLS:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    df["gene_is_expressed"] = df["gene_is_expressed"].astype(bool)
    df["is_self_promoter"] = df["is_self_promoter"].astype(bool)

    df_agg = aggregate_abc_per_enhancer_gene_celltype(df)
    abc_full = aggregate_abc_to_full_dict(df_agg, present_threshold, strong_threshold)

    print(f"  Built {len(abc_full)} enhancer-gene links")
    return abc_full


def map_abc_enhancers_to_ccres(
    df_abc: pd.DataFrame,
    regulatory_elements_path: str,
) -> pd.DataFrame:
    """Map ABC enhancers to cCREs using center-in-interval rule."""
    print("Mapping ABC enhancers to cCREs...")

    reg = pd.read_csv(regulatory_elements_path)
    reg["start"] = pd.to_numeric(reg["start"], errors="coerce")
    reg["end"] = pd.to_numeric(reg["end"], errors="coerce")
    reg = reg.dropna(subset=["start", "end"])
    reg = reg.rename(columns={"start": "reg_start", "end": "reg_end"})
    reg[["reg_start", "reg_end"]] = reg[["reg_start", "reg_end"]].astype("int64")

    abc = df_abc.copy()
    abc["start"] = pd.to_numeric(abc["start"], errors="coerce")
    abc["end"] = pd.to_numeric(abc["end"], errors="coerce")
    abc = abc.dropna(subset=["start", "end"])
    abc = abc.rename(columns={"start": "enh_start", "end": "enh_end"})
    abc[["enh_start", "enh_end"]] = abc[["enh_start", "enh_end"]].astype("int64")
    abc["enh_center"] = ((abc["enh_start"] + abc["enh_end"]) // 2).astype("int64")

    mapped_chunks = []
    for chrom, reg_chr in reg.groupby("chrom"):
        abc_chr = abc[abc["chrom"] == chrom]
        if abc_chr.empty:
            continue

        reg_chr = reg_chr.sort_values("reg_start").reset_index(drop=True)
        abc_chr = abc_chr.sort_values("enh_center").reset_index(drop=True)

        merged = pd.merge_asof(
            abc_chr, reg_chr,
            left_on="enh_center", right_on="reg_start",
            direction="backward",
        )

        inside = (merged["enh_center"] >= merged["reg_start"]) & (merged["enh_center"] < merged["reg_end"])
        merged = merged[inside]

        if not merged.empty:
            merged["chrom"] = chrom
            keep_cols = ["ENCODE_id", "gene_name", "ABC_full", "chrom", "enh_start", "enh_end"]
            mapped_chunks.append(merged[[c for c in keep_cols if c in merged.columns]])

    if not mapped_chunks:
        raise ValueError("No ABC enhancers could be mapped to cCREs.")

    result = pd.concat(mapped_chunks, ignore_index=True)
    result = result.drop_duplicates(
        subset=["ENCODE_id", "gene_name", "chrom", "enh_start", "enh_end"],
        keep="first",
    )

    print(f"  Mapped {len(result)} enhancer-cCRE pairs")
    return result


def collapse_abc_enhancers_per_link(df_mapped: pd.DataFrame) -> pd.DataFrame:
    """Collapse multiple ABC enhancers per (cCRE_id, gene_name) into a list."""

    def build_enhancer_list(df_sub: pd.DataFrame) -> Dict[str, List]:
        enhancers = []
        for _, r in df_sub.iterrows():
            enhancers.append({
                "start": int(r["enh_start"]),
                "end": int(r["enh_end"]),
                "ABC_full": r["ABC_full"] if "ABC_full" in df_sub.columns else {},
            })
        return {"ABC_enhancers": enhancers}

    return (
        df_mapped
        .groupby(["ENCODE_id", "gene_name"], as_index=False)
        .apply(lambda df_sub: pd.Series(build_enhancer_list(df_sub)))
    )


# =============================================================================
# FINAL ELEMENT TABLE (SCREEN + ABC)
# =============================================================================

def merge_all_link_sources(
    df_screen_exp: pd.DataFrame,
    df_screen_comp: pd.DataFrame,
    df_abc_collapsed: pd.DataFrame,
) -> pd.DataFrame:
    """Merge experimental, computational, and ABC links into unified table."""
    print("Merging all link sources...")

    for df in (df_screen_exp, df_screen_comp):
        for col in ["ENCODE_id", "gene_id", "gene_name", "gene_type"]:
            if col in df.columns:
                df[col] = df[col].astype(str)

    df_merged = df_screen_exp.merge(
        df_screen_comp,
        on=["ENCODE_id", "gene_id", "gene_name", "gene_type"],
        how="outer",
    )

    if "region_comp" in df_merged.columns or "region_3D" in df_merged.columns:
        df_merged["region"] = df_merged.get("region_comp", df_merged.get("region_3D"))
        df_merged = df_merged.drop(
            columns=[c for c in ["region_3D", "region_comp"] if c in df_merged.columns]
        )

    df_merged = df_merged.merge(df_abc_collapsed, on=["ENCODE_id", "gene_name"], how="left")

    print(f"  Merged table: {len(df_merged)} rows")
    return df_merged


def build_element_gene_links_table(
    df_links: pd.DataFrame,
    screen_exp_biosamples: List[str],
    screen_exp_assays: List[str],
    screen_comp_biosamples: List[str],
    screen_comp_assays: List[str],
) -> pd.DataFrame:
    """Build final element-level table with nested gene_links column."""
    print("Building element-gene links table...")

    def build_gene_links(df_element: pd.DataFrame) -> Dict[str, Dict]:
        out = {}

        for gene_name, df_gene in df_element.groupby("gene_name", sort=False):
            base = df_gene.iloc[0].to_dict()
            base.pop("ENCODE_id", None)
            base.pop("gene_name", None)

            base["screen_exp"] = ensure_screen_block(
                base.get("screen_exp"), screen_exp_biosamples, screen_exp_assays
            )
            base["screen_comp"] = ensure_screen_block(
                base.get("screen_comp"), screen_comp_biosamples, screen_comp_assays
            )

            if "ABC_enhancers" in df_gene.columns:
                enh_lists = [x for x in df_gene["ABC_enhancers"] if isinstance(x, (list, tuple)) and x]
                base["ABC_enhancers"] = [e for lst in enh_lists for e in lst] if enh_lists else []
            else:
                base["ABC_enhancers"] = []

            # Initialize empty hichip dict (will be populated later)
            base["hichip"] = {}

            out[gene_name] = base

        return out

    gene_links_series = (
        df_links
        .groupby("ENCODE_id", sort=False)
        .apply(build_gene_links)
        .rename("gene_links")
    )

    result = gene_links_series.reset_index()
    print(f"  Built {len(result)} element records")
    return result


# =============================================================================
# HiCHIP LOOP FILE LOADING & NORMALIZATION
# =============================================================================

def load_loops_unified(path: str) -> pd.DataFrame:
    """
    Normalize different HiChIP loop formats into a standard DataFrame:
        chr1, start1, end1, chr2, start2, end2, counts, score, n_reps
    """
    ext = os.path.splitext(path)[1].lower()

    if ext == ".bedpe":
        df = pd.read_csv(path, sep="\t")
    else:
        df = pd.read_csv(path, sep="\t", header=None)

    ncols = df.shape[1]

    if {"chr1", "start1", "end1", "chr2", "start2", "end2"}.issubset(df.columns):
        return _normalize_counts_score_from_bedpe(df, path)

    if ncols == 9:
        df.columns = ["chr1", "start1", "end1", "chr2", "start2", "end2", "counts", "score", "n_reps"]
        return _finalize_counts_score(df, path, source="bedpe_no_header")

    if ncols == 8:
        df.columns = ["chr1", "start1", "end1", "chr2", "start2", "end2", "counts", "n_reps"]
        df["score"] = np.nan
        return _finalize_counts_score(df, path, source="bedpe_no_score")

    if ncols == 4:
        return _parse_loopcatalog_bed(df, path)

    raise ValueError(f"Unknown loop format with {ncols} columns in {path}")


def _normalize_counts_score_from_bedpe(df: pd.DataFrame, path: str) -> pd.DataFrame:
    """Normalize BEDPE with header (HMEC/HiChIPper)."""
    coord_cols = ["chr1", "start1", "end1", "chr2", "start2", "end2"]
    missing = [c for c in coord_cols if c not in df.columns]
    if missing:
        raise ValueError(f"[bedpe] Missing coordinate columns in {path}: {missing}")

    if "PETs" in df.columns:
        counts = df["PETs"].astype(float)
    elif "weight" in df.columns:
        counts = df["weight"].astype(float)
    else:
        raise ValueError(f"[bedpe] No PETs/weight column in {path}.")

    out = df.copy()
    out["counts"] = counts
    
    if "n_reps" not in out.columns:
        out["n_reps"] = 1
    if "score" not in out.columns:
        out["score"] = np.nan

    return _finalize_counts_score(out, path, source="bedpe_with_header")


def _parse_loopcatalog_bed(df: pd.DataFrame, path: str) -> pd.DataFrame:
    """Parse LoopCatalog 4-column .bed format."""
    df.columns = ["chr1", "start1", "end1", "anchor2"]
    
    parsed = df["anchor2"].apply(lambda x: _parse_anchor2(x, path))
    df["chr2"] = parsed.str[0]
    df["start2"] = parsed.str[1].astype(int)
    df["end2"] = parsed.str[2].astype(int)
    df["score"] = parsed.str[3].astype(float)
    df["counts"] = np.nan
    df["n_reps"] = 1
    
    return _finalize_counts_score(df, path, source="loopcatalog_bed")


def _parse_anchor2(value: str, path: str) -> tuple:
    """Parse anchor2 format: chr2:start2-end2,score"""
    m = re.match(r"(chr[^:]+):(\d+)-(\d+),(.+)", str(value))
    if not m:
        raise ValueError(f"Unparseable anchor2 format in {path}: {value}")
    
    c2, s2, e2, v = m.groups()
    v = v.strip()
    score = float("inf") if v.lower() == "inf" else float(v)
    return c2, int(s2), int(e2), score


def _finalize_counts_score(df: pd.DataFrame, path: str, source: str) -> pd.DataFrame:
    """Ensure final types and column presence for loop data."""
    coord_cols = ["chr1", "start1", "end1", "chr2", "start2", "end2"]
    missing = [c for c in coord_cols if c not in df.columns]
    if missing:
        raise ValueError(f"[{source}] Missing coordinate columns in {path}: {missing}")

    out = df.copy()
    for col in ["counts", "score"]:
        if col not in out.columns:
            out[col] = np.nan
    if "n_reps" not in out.columns:
        out["n_reps"] = 1

    out["start1"] = out["start1"].astype(int)
    out["end1"] = out["end1"].astype(int)
    out["start2"] = out["start2"].astype(int)
    out["end2"] = out["end2"].astype(int)
    out["counts"] = out["counts"].astype(float)
    out["score"] = out["score"].astype(float)
    out["n_reps"] = out["n_reps"].astype(int)

    return out[["chr1", "start1", "end1", "chr2", "start2", "end2", "counts", "score", "n_reps"]]


# =============================================================================
# HiCHIP ANCHOR BUILDING
# =============================================================================

def build_anchors(loops: pd.DataFrame) -> pd.DataFrame:
    """Build anchor DataFrame from loops (one row per anchor A/B)."""
    if "loop_id" not in loops.columns:
        loops = loops.copy()
        loops["loop_id"] = np.arange(len(loops))

    anchor_a = loops[["chr1", "start1", "end1", "loop_id", "counts", "score", "n_reps"]].copy()
    anchor_a.rename(columns={"chr1": "chr", "start1": "start", "end1": "end"}, inplace=True)
    anchor_a["anchor"] = "A"

    anchor_b = loops[["chr2", "start2", "end2", "loop_id", "counts", "score", "n_reps"]].copy()
    anchor_b.rename(columns={"chr2": "chr", "start2": "start", "end2": "end"}, inplace=True)
    anchor_b["anchor"] = "B"

    return pd.concat([anchor_a, anchor_b], ignore_index=True)


# =============================================================================
# HiCHIP PYRANGES HELPERS
# =============================================================================

def _to_pyranges_coords(df: pd.DataFrame, chr_col: str = "chr") -> pd.DataFrame:
    """Rename coordinates for PyRanges compatibility."""
    rename_map = {chr_col: "Chromosome", "start": "Start", "end": "End"}
    if "chrom" in df.columns:
        rename_map["chrom"] = "Chromosome"
    return df.rename(columns=rename_map)


def _from_pyranges_coords(df: pd.DataFrame, chr_col: str = "chr") -> pd.DataFrame:
    """Rename coordinates back from PyRanges format."""
    rename_map = {"Chromosome": chr_col, "Start": "start", "End": "end"}
    return df.rename(columns=rename_map)


# =============================================================================
# HiCHIP FILE DISCOVERY
# =============================================================================

def find_hichip_loops_file(cell_type_hichip_dir: str, celltype: str, hichip_experiment: str) -> str:
    """Find the loops file for a given cell type and experiment."""
    candidates = [
        os.path.join(cell_type_hichip_dir, f"{celltype}_{hichip_experiment}_loops_hg38.bedpe"),
        os.path.join(cell_type_hichip_dir, f"{celltype}_{hichip_experiment}_loops_hg38.bed"),
    ]
    
    for c in candidates:
        if os.path.exists(c):
            return c

    pattern = os.path.join(cell_type_hichip_dir, "*_loops_hg38.bed*")
    matches = glob.glob(pattern)
    if matches:
        return matches[0]

    existing = os.listdir(cell_type_hichip_dir) if os.path.exists(cell_type_hichip_dir) else []
    raise FileNotFoundError(
        f"No HiChIP loops file found for {celltype} / {hichip_experiment}.\n"
        f"Looked in: {cell_type_hichip_dir}\n"
        f"Existing files: {existing}"
    )


# =============================================================================
# HiCHIP OVERLAP COMPUTATION
# =============================================================================

def compute_hichip_overlaps(
    loops: pd.DataFrame,
    ccre_df: pd.DataFrame,
    genes_df: pd.DataFrame
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Compute anchor overlaps with cCREs and promoters.
    
    Returns:
        tuple: (anchor_metrics, overlap_ccre, overlap_prom)
    """
    

    loops = loops.copy()
    if "loop_id" not in loops.columns:
        loops["loop_id"] = np.arange(len(loops))
    
    anchors = build_anchors(loops)
    
    # Separate metrics (contains NaN/inf that PyRanges can't handle)
    anchor_metrics = anchors[["loop_id", "anchor", "counts", "score", "n_reps"]].drop_duplicates()
    anchors_for_pr = anchors[["chr", "start", "end", "loop_id", "anchor"]].copy()
    
    # Prepare promoters
    promoters = genes_df[["chrom", "prom_start", "prom_end", "gene_id", "gene_name"]].copy()
    promoters.rename(columns={"prom_start": "start", "prom_end": "end"}, inplace=True)
    
    # Prepare cCREs - only keep necessary columns to avoid duplicates after join
    ccre_for_pr = ccre_df[["chrom", "start", "end", "cCRE_id"]].copy()
    
    # Convert to PyRanges format
    anchors_pr = _to_pyranges_coords(anchors_for_pr, chr_col="chr")
    promoters_pr = _to_pyranges_coords(promoters, chr_col="chrom")
    ccre_pr = _to_pyranges_coords(ccre_for_pr, chr_col="chrom")
    
    # Compute overlaps
    gr_anchors = pr.PyRanges(anchors_pr)
    gr_promoters = pr.PyRanges(promoters_pr)
    gr_ccres = pr.PyRanges(ccre_pr)
    
    overlap_prom = gr_anchors.join(gr_promoters).df
    overlap_ccre = gr_anchors.join(gr_ccres).df
    
    # Rejoin metrics
    overlap_prom = overlap_prom.merge(anchor_metrics, on=["loop_id", "anchor"], how="left")
    overlap_ccre = overlap_ccre.merge(anchor_metrics, on=["loop_id", "anchor"], how="left")
    
    return anchor_metrics, overlap_ccre, overlap_prom

# =============================================================================
# HiCHIP CELL LINE COLUMN INTEGRATION
# =============================================================================

def integrate_hichip_per_cell_line(
    reg_table: pd.DataFrame,
    loops_df: pd.DataFrame,
    overlap_ccre: pd.DataFrame,
    overlap_prom: pd.DataFrame,
    cell_line: str,
    hichip_key: str = "hichip",
) -> pd.DataFrame:
    """Add nested 'hichip' entry into reg_table[cell_line] dicts, per cCRE."""
    
    _validate_hichip_overlap_columns(overlap_ccre, overlap_prom)
    
    if "loop_id" not in loops_df.columns:
        loops_df = loops_df.copy()
        loops_df["loop_id"] = np.arange(len(loops_df))
    
    loop_coords = _build_loop_coords(loops_df)
    loop_to_ccres = _build_loop_to_ccres(overlap_ccre)
    loop_to_genes = _build_loop_to_genes(overlap_prom)
    per_ccre = _build_per_ccre_hichip(overlap_ccre, loop_coords, loop_to_ccres, loop_to_genes)
    
    if cell_line not in reg_table.columns:
        reg_table[cell_line] = [{} for _ in range(len(reg_table))]
    else:
        reg_table[cell_line] = reg_table[cell_line].apply(
            lambda v: v if isinstance(v, dict) else {}
        )
    
    def add_hichip(row):
        cell_dict = row[cell_line]
        hdata = per_ccre.get(
            str(row["cCRE_id"]),
            {"n_loops": 0, "max_counts": None, "max_score": None, "loops": []}
        )
        cell_dict[hichip_key] = hdata
        return cell_dict
    
    reg_table[cell_line] = reg_table.apply(add_hichip, axis=1)
    return reg_table


def _validate_hichip_overlap_columns(overlap_ccre: pd.DataFrame, overlap_prom: pd.DataFrame):
    """Validate required columns in HiChIP overlap DataFrames."""
    required_ccre = {"cCRE_id", "loop_id", "anchor", "counts", "score", "n_reps"}
    missing = required_ccre - set(overlap_ccre.columns)
    if missing:
        raise ValueError(f"overlap_ccre missing columns: {missing}")
    
    required_prom = {"loop_id", "anchor", "gene_id", "gene_name"}
    missing = required_prom - set(overlap_prom.columns)
    if missing:
        raise ValueError(f"overlap_prom missing columns: {missing}")


def _build_loop_coords(loops_df: pd.DataFrame) -> dict:
    """Build loop_id -> anchor coordinates mapping."""
    return (
        loops_df.set_index("loop_id")
        .apply(lambda r: {
            "anchor_a": {"chr": r["chr1"], "start": int(r["start1"]), "end": int(r["end1"])},
            "anchor_b": {"chr": r["chr2"], "start": int(r["start2"]), "end": int(r["end2"])},
        }, axis=1)
        .to_dict()
    )


def _build_loop_to_ccres(overlap_ccre: pd.DataFrame) -> dict:
    """Build loop_id -> set of cCRE_ids mapping."""
    return (
        overlap_ccre.groupby("loop_id")["cCRE_id"]
        .apply(lambda s: set(s.astype(str)))
        .to_dict()
    )


def _build_loop_to_genes(overlap_prom: pd.DataFrame) -> dict:
    """Build loop_id -> {anchor -> [genes]} mapping."""
    loop_to_genes = defaultdict(lambda: defaultdict(list))
    for _, r in overlap_prom.iterrows():
        loop_to_genes[r["loop_id"]][r["anchor"]].append({
            "gene_id": str(r["gene_id"]),
            "gene_name": str(r["gene_name"]),
        })
    return loop_to_genes


def _build_per_ccre_hichip(
    overlap_ccre: pd.DataFrame,
    loop_coords: dict,
    loop_to_ccres: dict,
    loop_to_genes: dict
) -> dict:
    """Build per-cCRE HiChIP summary dictionaries."""
    per_ccre = {}
    
    for ccre_id, df_cc in overlap_ccre.groupby("cCRE_id"):
        seen_loops = set()
        loops_list = []
        max_counts = None
        max_score = None
        ccre_id_str = str(ccre_id)
        
        for _, row in df_cc.iterrows():
            lid = row["loop_id"]
            if lid in seen_loops:
                continue
            seen_loops.add(lid)
            
            cnt = float(row["counts"]) if pd.notna(row["counts"]) else None
            scr = float(row["score"]) if pd.notna(row["score"]) else None
            
            if cnt is not None:
                max_counts = cnt if max_counts is None else max(max_counts, cnt)
            if scr is not None and np.isfinite(scr):
                max_score = scr if max_score is None else max(max_score, scr)
            
            ccre_anchor = row["anchor"]
            partner_anchor = "B" if ccre_anchor == "A" else "A"
            
            partner_genes = [
                {"gene_name": g["gene_name"], "gene_id": g["gene_id"], "anchor": partner_anchor}
                for g in loop_to_genes.get(lid, {}).get(partner_anchor, [])
            ]
            
            base = {
                "loop_id": int(lid),
                "counts": cnt,
                "score": scr,
                "n_reps": int(row["n_reps"]),
                "partner_genes": partner_genes,
                "partner_cCREs": sorted(loop_to_ccres.get(lid, set()) - {ccre_id_str}),
            }
            
            coords = loop_coords.get(lid)
            if coords:
                base.update(coords)
            
            loops_list.append(base)
        
        if loops_list:
            per_ccre[ccre_id_str] = {
                "n_loops": len(loops_list),
                "max_counts": max_counts,
                "max_score": max_score,
                "loops": loops_list,
            }
    
    return per_ccre


# =============================================================================
# HiCHIP GENE LINKS INTEGRATION
# =============================================================================

def build_ccre_gene_hichip_map(
    overlap_ccre: pd.DataFrame,
    overlap_prom: pd.DataFrame
) -> dict:
    """Build cCRE -> gene -> [loops] mapping for opposite-anchor pairs."""
    cc = overlap_ccre.rename(columns={"anchor": "anchor_ccre"})
    cp = overlap_prom.rename(columns={"anchor": "anchor_prom"})
    
    merged = cc.merge(cp, on="loop_id", suffixes=("_ccre", "_prom"))
    merged_ep = merged[merged["anchor_ccre"] != merged["anchor_prom"]]
    
    cg = defaultdict(lambda: defaultdict(list))
    for _, r in merged_ep.iterrows():
        cg[r["cCRE_id"]][r["gene_name"]].append({
            "loop_id": r["loop_id"],
            "counts": float(r["counts_ccre"]) if pd.notna(r["counts_ccre"]) else None,
            "score": float(r["score_ccre"]) if pd.notna(r["score_ccre"]) else None,
            "n_reps": r["n_reps_ccre"],
            "anchor_ccre": r["anchor_ccre"],
            "anchor_prom": r["anchor_prom"],
        })
    
    return cg


def add_hichip_to_gene_links(
    gene_links: dict,
    ccre_id: str,
    cell_line: str,
    ccre_gene_map: dict,
) -> dict:
    """
    Add HiChIP data to gene_links dict.
    
    Each gene keeps a 'hichip' dict. A cell_line entry appears only if there are loops.
    """
    per_gene_loops = ccre_gene_map.get(ccre_id, {})

    for gene_name, gdict in gene_links.items():
        hichip = gdict.setdefault("hichip", {})
        loops_for_pair = per_gene_loops.get(gene_name, [])

        if not loops_for_pair:
            continue

        counts_vals = [l.get("counts") for l in loops_for_pair if l.get("counts") is not None]
        score_vals = [l.get("score") for l in loops_for_pair if l.get("score") is not None and np.isfinite(l.get("score"))]

        hichip[cell_line] = {
            "n_loops": len(loops_for_pair),
            "max_counts": max(counts_vals) if counts_vals else None,
            "max_score": max(score_vals) if score_vals else None,
            "loops": loops_for_pair,
        }

    return gene_links


# =============================================================================
# HiCHIP MAIN PROCESSING FUNCTIONS
# =============================================================================

def handle_cell_type_hichip(
    cell_type_hichip_dir: str,
    hichip_experiment: str,
    celltype: str,
    ccre_df: pd.DataFrame,
    genes_df: pd.DataFrame
) -> pd.DataFrame:
    """Process HiChIP data for a single cell type."""
    loops_path = find_hichip_loops_file(cell_type_hichip_dir, celltype, hichip_experiment)
    print(f"[HiChIP] Using loops file for {celltype}: {loops_path}")
    
    loops = load_loops_unified(loops_path)
    loops["loop_id"] = np.arange(len(loops))
    
    _, overlap_ccre, overlap_prom = compute_hichip_overlaps(loops, ccre_df, genes_df)
    
    ccre_df = integrate_hichip_per_cell_line(ccre_df, loops, overlap_ccre, overlap_prom, celltype)
    
    ccre_gene_map = build_ccre_gene_hichip_map(overlap_ccre, overlap_prom)
    
    ccre_df["gene_links"] = ccre_df.apply(
        lambda row: add_hichip_to_gene_links(
            row["gene_links"], row["cCRE_id"], celltype, ccre_gene_map
        ) if pd.notna(row["gene_links"]) and isinstance(row["gene_links"], dict) else row["gene_links"],
        axis=1
    )
    
    return ccre_df


def merge_all_hichips(
    hichip_dir: str,
    hichip_experiment: str,
    hichip_panel: List[str],
    ccre_df: pd.DataFrame,
    genes_df: pd.DataFrame
) -> pd.DataFrame:
    """Process HiChIP data for all cell types in panel."""
    for celltype in hichip_panel:
        cell_type_hichip_dir = os.path.join(hichip_dir, celltype)
        if os.path.exists(cell_type_hichip_dir):
            ccre_df = handle_cell_type_hichip(
                cell_type_hichip_dir, hichip_experiment, celltype, ccre_df, genes_df
            )
        else:
            print(f"[HiChIP] Skipping {celltype}: directory not found")
    
    return ccre_df


# =============================================================================
# MAIN EXECUTION
# =============================================================================

if __name__ == "__main__":
    # =========================================================================
    # CONFIGURATION
    # =========================================================================

    # Paths
    SCREEN_EXP_ZIP = r"C:\Users\stavz\Downloads\Human-Gene-Links.zip"
    SCREEN_EXP_FILE = "V4-hg38.Gene-Links.3D-Chromatin.txt"
    SCREEN_COMP_PATH = r"C:\Users\stavz\Downloads\V4-hg38.Gene-Links.Computational-Methods.txt.gz"
    REG_ELEMENTS_PATH = r"C:\Users\stavz\Desktop\masters\APM\test\regulatory_elements_matching\regulatory_element_focus.csv"
    ABC_PATH = r"C:\Users\stavz\Downloads\AllPredictions.AvgHiC.ABC0.015.minus150.ForABCPaperV3.txt"
    HICHIP_DIR = r'C:\Users\stavz\Desktop\masters\APM\data\HiCHIP'

    # Gene panel
    GENE_PANEL = [
        "PSMB8", "PSMB9", "PSMB10", "PSME1", "PSME2", "PSME4",
        "TAP1", "TAP2", "TAPBP", "TAPBPL", "CALR", "PDIA3", "PDIA6", "B2M",
        "HLA-A", "HLA-B", "HLA-C", "HLA-E", "HLA-F", "HLA-G",
        "MICA", "MICB", "ULBP1", "ULBP2", "ULBP3", "RAET1E", "RAET1G", "RAET1L",
        "PVR", "NECTIN2", "NCR3LG1", "SOCS1", "SOCS3", "NFKBIA", "TNFAIP3",
        "STAT1", "STAT3", "JAK1", "JAK2", "IRF1", "IRF2", "NLRC5",
        "CD274", "PDCD1LG2", "IFNG", "IFNGR1", "IFNGR2", "ADAM10", "ADAM17",
        "TGFB1", "IL6", "IL6R", "IL6ST", "PIAS1", "PIAS3", "PTPN2", "PTPN11",
        "EGFR", "SRC", "SMAD3", "EP300", "CREBBP", "CCL5", "CXCL9", "CXCL10", "CXCL11",
    ]

    # Biosamples / cell types
    BIOSAMPLES_SCREEN_EXP = [
        "MCF-7", "MCF_10A", "Breast Mammary Tissue",
        "breast_epithelium_tissue_female_adult_(53_years)",
        "mammary_epithelial_cell_female_adult_(19_years)",
    ]

    BIOSAMPLES_SCREEN_COMP = [
        "MCF-7", "MCF_10A", "T47D", "mammary_epithelial_cell", "breast_epithelium",
    ]

    CELLTYPES_ABC = [
        "MCF-7-ENCODE", "MCF10A-Ji2017", "MCF10A_treated_with_TAM24hr-Ji2017",
        "MDA-MB-231", "mammary_epithelial_cell-Roadmap", "breast_epithelium-ENCODE",
    ]

    HICHIP_PANEL = ["B80T5", "HMEC", "K5plusK19plus", "MCF7", "MDA-MB-231", "T47D", "ZR751"]
    HICHIP_EXPERIMENT = "H3K27ac"

    # =========================================================================
    # STEP 1: BUILD SCREEN LINKS
    # =========================================================================
    print("\n" + "=" * 60)
    print("STEP 1: Building SCREEN links")
    print("=" * 60)

    # SCREEN experimental (3D chromatin)
    links_exp, assays_exp = build_screen_exp_links(
        zip_path=SCREEN_EXP_ZIP,
        inner_file=SCREEN_EXP_FILE,
        gene_list=GENE_PANEL,
        panel_biosamples=BIOSAMPLES_SCREEN_EXP,
        breast_biosamples=BIOSAMPLES_SCREEN_EXP,
        intact_pvalue_threshold=1e-6,
    )
    links_exp = collapse_screen_to_nested(links_exp, BIOSAMPLES_SCREEN_EXP, assays_exp, "screen_exp")

    # SCREEN computational
    links_comp, assays_comp = build_screen_comp_links(
        gz_path=SCREEN_COMP_PATH,
        gene_list=GENE_PANEL,
        panel_biosamples=BIOSAMPLES_SCREEN_COMP,
        breast_biosamples=BIOSAMPLES_SCREEN_COMP,
    )
    links_comp = collapse_screen_to_nested(links_comp, BIOSAMPLES_SCREEN_COMP, assays_comp, "screen_comp")

    # =========================================================================
    # STEP 2: BUILD ABC LINKS
    # =========================================================================
    print("\n" + "=" * 60)
    print("STEP 2: Building ABC links")
    print("=" * 60)

    abc_raw = build_abc_enhancer_links(
        abc_path=ABC_PATH,
        gene_list=GENE_PANEL,
        celltypes=CELLTYPES_ABC,
    )
    abc_mapped = map_abc_enhancers_to_ccres(abc_raw, REG_ELEMENTS_PATH)
    abc_collapsed = collapse_abc_enhancers_per_link(abc_mapped)

    # =========================================================================
    # STEP 3: MERGE ALL SOURCES AND BUILD ELEMENT TABLE
    # =========================================================================
    print("\n" + "=" * 60)
    print("STEP 3: Merging sources and building element table")
    print("=" * 60)

    links_merged = merge_all_link_sources(links_exp, links_comp, abc_collapsed)

    elements = build_element_gene_links_table(
        df_links=links_merged,
        screen_exp_biosamples=BIOSAMPLES_SCREEN_EXP,
        screen_exp_assays=assays_exp,
        screen_comp_biosamples=BIOSAMPLES_SCREEN_COMP,
        screen_comp_assays=assays_comp,
    )

    # Merge with regulatory elements
    reg_df = pd.read_csv(REG_ELEMENTS_PATH)
    reg_df = reg_df.merge(elements, on="ENCODE_id", how="left")

    # =========================================================================
    # STEP 4: INTEGRATE HiCHIP DATA
    # =========================================================================
    print("\n" + "=" * 60)
    print("STEP 4: Integrating HiChIP data")
    print("=" * 60)

    # Prepare for HiChIP integration
    # reg_df.rename(columns={"ENCODE_id": "cCRE_id"}, inplace=True)

    # Load genes_df (you'll need to define this based on your data)
    # genes_df should have columns: chrom, prom_start, prom_end, gene_id, gene_name
    # Example placeholder - replace with actual gene data loading:
    # genes_df = pd.read_csv(r"C:\Users\stavz\Desktop\masters\APM\test\primary_genes_only.csv")
    
    # For now, assuming genes_df is already defined elsewhere
    # Uncomment and modify the following when genes_df is available:
    reg_df = merge_all_hichips(HICHIP_DIR, HICHIP_EXPERIMENT, HICHIP_PANEL, reg_df, genes_df)

    # =========================================================================
    # VERIFICATION
    # =========================================================================
    print("\n" + "=" * 60)
    print("VERIFICATION")
    print("=" * 60)

    print(f"Final regulatory elements table: {len(reg_df)} records")
    print(f"Columns: {list(reg_df.columns)}")
    
    # Check gene_links structure
    sample_idx = reg_df["gene_links"].notna().idxmax()
    if pd.notna(reg_df.loc[sample_idx, "gene_links"]):
        sample = reg_df.loc[sample_idx, "gene_links"]
        sample_gene = list(sample.keys())[0]
        print(f"\nSample gene_links structure for '{sample_gene}':")
        print(f"  Top-level keys: {list(sample[sample_gene].keys())}")
        if "screen_exp" in sample[sample_gene]:
            print(f"  screen_exp keys: {list(sample[sample_gene]['screen_exp'].keys())}")
        if "hichip" in sample[sample_gene]:
            print(f"  hichip keys: {list(sample[sample_gene]['hichip'].keys())}")


STEP 1: Building SCREEN links
Building SCREEN experimental links...
  Chunk 10: kept 1597 rows
  Total rows extracted: 14841
  Assay types found: ['RNAPII-ChIAPET']
  Per-assay score quantiles:
                    q_weak  q_strong
    assay_type                      
    RNAPII-ChIAPET    56.5     87.72


C:\Users\stavz\AppData\Local\Temp\ipykernel_572\4150611443.py:418: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  bio_series = df_bio.groupby(link_cols).apply(make_assay_dict)
C:\Users\stavz\AppData\Local\Temp\ipykernel_572\4150611443.py:460: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return cons.groupby(link_cols).apply(to_dict).rename(column_name).reset_index()
C:\Users\stavz\AppData\Local\Temp\ipykernel_572\4150

  Built 4988 links
Building SCREEN computational links...
  Chunk 10: kept 1581 rows
  Total rows extracted: 12080
  Assay types found: ['ABC_(DNase_only)', 'EPIraction', 'rE2G_(DNase_only)']
  Per-assay score quantiles:
                         q_weak  q_strong
    assay_type                           
    ABC_(DNase_only)   1.000000  1.000000
    EPIraction         0.059618  0.084092
    rE2G_(DNase_only)  0.999993  0.999997


C:\Users\stavz\AppData\Local\Temp\ipykernel_572\4150611443.py:460: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return cons.groupby(link_cols).apply(to_dict).rename(column_name).reset_index()


  Built 3947 links

STEP 2: Building ABC links
Building ABC enhancer links...
  Chunk 10: kept 44 rows
  Total rows extracted: 1570


C:\Users\stavz\AppData\Local\Temp\ipykernel_572\4150611443.py:377: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(build_abc_full)


  Built 1568 enhancer-gene links
Mapping ABC enhancers to cCREs...
  Mapped 297 enhancer-cCRE pairs

STEP 3: Merging sources and building element table
Merging all link sources...
  Merged table: 7835 rows
Building element-gene links table...


C:\Users\stavz\AppData\Local\Temp\ipykernel_572\4150611443.py:732: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda df_sub: pd.Series(build_enhancer_list(df_sub)))
C:\Users\stavz\AppData\Local\Temp\ipykernel_572\4150611443.py:812: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(build_gene_links)


  Built 6949 element records

STEP 4: Integrating HiChIP data
[HiChIP] Using loops file for B80T5: C:\Users\stavz\Desktop\masters\APM\data\HiCHIP\B80T5\B80T5_H3K27ac_loops_hg38.bed
[HiChIP] Using loops file for HMEC: C:\Users\stavz\Desktop\masters\APM\data\HiCHIP\HMEC\HMEC_H3K27ac_loops_hg38.bedpe
[HiChIP] Using loops file for K5plusK19plus: C:\Users\stavz\Desktop\masters\APM\data\HiCHIP\K5plusK19plus\K5plusK19plus_H3K27ac_loops_hg38.bed
[HiChIP] Using loops file for MCF7: C:\Users\stavz\Desktop\masters\APM\data\HiCHIP\MCF7\MCF7_H3K27ac_loops_hg38.bedpe
[HiChIP] Using loops file for MDA-MB-231: C:\Users\stavz\Desktop\masters\APM\data\HiCHIP\MDA-MB-231\MDA-MB-231_H3K27ac_loops_hg38.bed
[HiChIP] Using loops file for T47D: C:\Users\stavz\Desktop\masters\APM\data\HiCHIP\T47D\T47D_H3K27ac_loops_hg38.bed
[HiChIP] Using loops file for ZR751: C:\Users\stavz\Desktop\masters\APM\data\HiCHIP\ZR751\ZR751_H3K27ac_loops_hg38.bedpe

VERIFICATION
Final regulatory elements table: 49921 records
Columns:

In [16]:
reg_df[reg_df["gene_links"].notna()]

,Unnamed: 0,cCRE_id,ENCODE_id,type,raw_type,chrom,start,end,0–100kb,100–250kb,...,genes_by_exact_dist,HMEC1,MCF7,gene_links,B80T5,HMEC,K5plusK19plus,MDA-MB-231,T47D,ZR751
164,164,EH38D2176929,EH38E1355325,dELS,"dELS,CTCF-bound",chr1,64824161,64824505,NaN,JAK1,...,JAK1:243249,"{'in_HMEC1': True, 'H3K27ac': None, 'H3K4me3':...","{'hichip': {'n_loops': 0, 'max_counts': None, ...","{'JAK1': {'gene_id': 'ENSG00000162434', 'gene_...","{'hichip': {'n_loops': 0, 'max_counts': None, ...","{'hichip': {'n_loops': 2, 'max_counts': 11.0, ...","{'hichip': {'n_loops': 0, 'max_counts': None, ...","{'hichip': {'n_loops': 0, 'max_counts': None, ...","{'hichip': {'n_loops': 0, 'max_counts': None, ...","{'hichip': {'n_loops': 0, 'max_counts': None, ..."
167,167,EH38D2176966,EH38E1355356,dELS,"dELS,CTCF-bound",chr1,64864102,64864452,NaN,JAK1,...,JAK1:203302,"{'in_HMEC1': True, 'H3K27ac': None, 'H3K4me3':...","{'hichip': {'n_loops': 0, 'max_counts': None, ...","{'JAK1': {'gene_id': 'ENSG00000162434', 'gene_...","{'hichip': {'n_loops': 0, 'max_counts': None, ...","{'hichip': {'n_loops': 0, 'max_counts': None, ...","{'hichip': {'n_loops': 10, 'max_counts': None,...","{'hichip': {'n_loops': 0, 'max_counts': None, ...","{'hichip': {'n_loops': 0, 'max_counts': None, ...","{'hichip': {'n_loops': 0, 'max_counts': None, ..."
168,168,EH38D2176967,EH38E1355357,dELS,dELS,chr1,64864583,64864849,NaN,JAK1,...,JAK1:202905,"{'in_HMEC1': True, 'H3K27ac': None, 'H3K4me3':...","{'hichip': {'n_loops': 0, 'max_counts': None, ...","{'JAK1': {'gene_id': 'ENSG00000162434', 'gene_...","{'hichip': {'n_loops': 0, 'max_counts': None, ...","{'hichip': {'n_loops': 0, 'max_counts': None, ...","{'hichip': {'n_loops': 10, 'max_counts': None,...","{'hichip': {'n_loops': 0, 'max_counts': None, ...","{'hichip': {'n_loops': 0, 'max_counts': None, ...","{'hichip': {'n_loops': 0, 'max_counts': None, ..."
173,173,EH38D2176987,EH38E1355375,dELS,"dELS,CTCF-bound",chr1,64878905,64879214,NaN,JAK1,...,JAK1:188540,"{'in_HMEC1': True, 'H3K27ac': None, 'H3K4me3':...","{'hichip': {'n_loops': 0, 'max_counts': None, ...","{'JAK1': {'gene_id': 'ENSG00000162434', 'gene_...","{'hichip': {'n_loops': 0, 'max_counts': None, ...","{'hichip': {'n_loops': 0, 'max_counts': None, ...","{'hichip': {'n_loops': 0, 'max_counts': None, ...","{'hichip': {'n_loops': 0, 'max_counts': None, ...","{'hichip': {'n_loops': 0, 'max_counts': None, ...","{'hichip': {'n_loops': 0, 'max_counts': None, ..."
174,174,EH38D2176988,EH38E1355376,dELS,dELS,chr1,64879218,64879564,NaN,JAK1,...,JAK1:188190,"{'in_HMEC1': True, 'H3K27ac': None, 'H3K4me3':...","{'hichip': {'n_loops': 0, 'max_counts': None, ...","{'JAK1': {'gene_id': 'ENSG00000162434', 'gene_...","{'hichip': {'n_loops': 0, 'max_counts': None, ...","{'hichip': {'n_loops': 0, 'max_counts': None, ...","{'hichip': {'n_loops': 0, 'max_counts': None, ...","{'hichip': {'n_loops': 0, 'max_counts': None, ...","{'hichip': {'n_loops': 0, 'max_counts': None, ...","{'hichip': {'n_loops': 0, 'max_counts': None, ..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49758,49758,EH38D6010243,EH38E3873031,pELS,"pELS,CTCF-bound",chr9,5629285,5629550,NaN,"CD274,PDCD1LG2",...,"PDCD1LG2:118794,CD274:178782,JAK2:644895","{'in_HMEC1': True, 'H3K27ac': None, 'H3K4me3':...","{'hichip': {'n_loops': 3, 'max_counts': 2.0, '...","{'JAK2': {'gene_id': 'ENSG00000096968', 'gene_...","{'hichip': {'n_loops': 48, 'max_counts': None,...","{'hichip': {'n_loops': 1, 'max_counts': 14.0, ...","{'hichip': {'n_loops': 28, 'max_counts': None,...","{'hichip': {'n_loops': 0, 'max_counts': None, ...","{'hichip': {'n_loops': 0, 'max_counts': None, ...","{'hichip': {'n_loops': 2, 'max_counts': 2.0, '..."
49759,49759,EH38D6010244,EH38E3873032,PLS,"PLS,CTCF-bound",chr9,5629625,5629949,NaN,"CD274,PDCD1LG2",...,"PDCD1LG2:119134,CD274:179122,JAK2:645235","{'in_HMEC1': True, 'H3K27ac': None, 'H3K4me3':...","{'hichip': {'n_loops': 3, 'max_counts': 2.0, '...","{'CD274': {'gene_id': 'ENSG0000

In [ ]:
import pandas as pd
REG_ELEMENTS_PATH = r"C:\Users\stavz\Desktop\masters\APM\test\regulatory_elements_matching\regulatory_element_focus.csv"

reg_df = pd.read_csv(REG_ELEMENTS_PATH)


In [13]:
import pandas as pd
genes_df = pd.read_csv(r'C:\Users\stavz\Desktop\masters\APM\data\primary_genes_only.csv')
def build_promoter_columns(genes: pd.DataFrame, tss_up: int = 2000, tss_down: int = 500) -> pd.DataFrame:
    genes["prom_start"] = genes.apply(lambda x: max(x["start"] - tss_up, 0) if x["strand"] == "+" else x["end"] + tss_up, axis=1) # to add chromosme end to the other side
    genes["prom_end"] = genes.apply(lambda x: min(x["start"] + tss_down, x["end"]) if x["strand"] == "+" else max(x["start"] - tss_down, 0), axis=1)
    return genes

genes_df = build_promoter_columns(genes_df, tss_up=2000, tss_down=500)

In [ ]:
"TCGA_HiChIP_FitHiChIP_interactions_CNV_corrected_norm_counts.txt"

In [2]:
import pandas as pd
import os
import sys

sys.path.append(r"C:\Users\stavz\Desktop\masters\APM")

from pipeline.config import PATHS

PATHS.working_dir



PosixPath('/home/stavz/masters/gdc/APM/data')

In [6]:
rna = pd.read_csv(PATHS.rna_expression_sample_metadata, sep = "\t")

In [16]:
rna

,raw_sample_id,participant,sample,sample_vial,aliquot,source
0,TCGA-AR-A5QQ-01,TCGA-AR-A5QQ,TCGA-AR-A5QQ-01,NaN,NaN,RNAexp_TCGA
1,TCGA-D8-A1JA-01,TCGA-D8-A1JA,TCGA-D8-A1JA-01,NaN,NaN,RNAexp_TCGA
2,TCGA-BH-A0BQ-01,TCGA-BH-A0BQ,TCGA-BH-A0BQ-01,NaN,NaN,RNAexp_TCGA
3,TCGA-BH-A0BT-01,TCGA-BH-A0BT,TCGA-BH-A0BT-01,NaN,NaN,RNAexp_TCGA
4,TCGA-A8-A06X-01,TCGA-A8-A06X,TCGA-A8-A06X-01,NaN,NaN,RNAexp_TCGA
...,...,...,...,...,...,...
1213,TCGA-A2-A3XT-01,TCGA-A2-A3XT,TCGA-A2-A3XT-01,NaN,NaN,RNAexp_TCGA
1214,TCGA-B6-A0X7-01,TCGA-B6-A0X7,TCGA-B6-A0X7-01,NaN,NaN,RNAexp_TCGA
1215,TCGA-BH-A1EV-11,TCGA-BH-A1EV,TCGA-BH-A1EV-11,NaN,NaN,RNAexp_TCGA
1216,TCGA-3C-AALJ-01,TCGA-3C-AALJ,TCGA-3C-AALJ-01,NaN,NaN,RNAexp_TCGA


In [13]:
presenence = pd.read_csv(r"analysis/sample_coverage/output/run_20260417_160851/tables/module_counts.tsv", sep = "\t")

In [15]:
presenence

,module,n_sample_vial,n_participant,n_primary_tumor_01,n_normal_blood_derived_10,n_normal_solid_tissue_11,n_normal_total_10_plus_11,n_other_or_unknown,sample_type_notes
0,SNV,927.0,924,925.0,919.0,5.0,924.0,2.0,NaN
1,SV,930.0,927,928.0,922.0,5.0,927.0,2.0,NaN
2,CNV,1060.0,1060,1060.0,975.0,85.0,1060.0,0.0,NaN
3,Methylation,893.0,791,791.0,0.0,97.0,97.0,5.0,NaN
4,RPPA,919.0,881,881.0,0.0,33.0,33.0,5.0,NaN
5,miRNA,832.0,750,749.0,0.0,76.0,76.0,7.0,NaN
6,RNA,1218.0,1097,1097.0,0.0,114.0,114.0,7.0,NaN
7,ATAC,74.0,74,74.0,0.0,0.0,0.0,0.0,NaN
8,HLA,1211.0,1081,1092.0,0.0,113.0,113.0,6.0,NaN
9,HiCHIP,NaN,13,13.0,0.0,0.0,0.0,0.0,participant_only_assumed_primary_tumor


In [ ]:
import json
import pandas as pd

with open("data/encode_tf_chip.json") as f:
    data = json.load(f)

experiments = data.get("@graph", [])

records = []
for exp in experiments:
    accession = exp.get("accession")

    target = exp.get("target", {})
    target_label = target.get("label") if isinstance(target, dict) else None

    biosample_obj = exp.get("biosample_ontology", {})
    biosample = biosample_obj.get("term_name") if isinstance(biosample_obj, dict) else None

    if not target_label or not biosample:
        continue

    records.append({
        "accession": accession,
        "target": target_label,
        "biosample": biosample,
        "assay_title": exp.get("assay_title"),
        "status": exp.get("status"),
    })

df = pd.DataFrame(records)
# df.to_csv("encode_tf_chip_experiments_long.csv", index=False)

# presence_matrix = (
#     df.assign(value=1)
#       .pivot_table(
#           index="target",
#           columns="biosample",
#           values="value",
#           aggfunc="max",
#           fill_value=0
#       )
# )
# presence_matrix.to_csv("encode_tf_chip_presence_matrix.csv")

# count_matrix = pd.pivot_table(
#     df,
#     index="target",
#     columns="biosample",
#     values="accession",
#     aggfunc="count",
#     fill_value=0
# )
# count_matrix.to_csv("encode_tf_chip_count_matrix.csv")

# accession_matrix = pd.pivot_table(
#     df,
#     index="target",
#     columns="biosample",
#     values="accession",
#     aggfunc=lambda x: ";".join(sorted(set(x)))
# )
# accession_matrix.to_csv("encode_tf_chip_accessions_matrix.csv")

# print(f"Experiments parsed: {len(df)}")
# print(f"Targets: {df['target'].nunique()}")
# print(f"Biosamples: {df['biosample'].nunique()}")

In [4]:
len(df["biosample"].unique())

218

In [12]:
cells = pd.read_csv("data/CHIP/CHIP_ATLAS/STAT1.bed", sep = "\t")

In [14]:
cells["experiment"].unique()

array(['SRX1799015'], dtype=object)

In [ ]:
import json
import pandas as pd

with open("data/encode_histone_chip.json") as f:
    data = json.load(f)

experiments = data["@graph"]

records = []

for exp in experiments:
    accession = exp.get("accession")

    # target (histone mark or TF)
    target = exp.get("target", {})
    target_label = target.get("label") if isinstance(target, dict) else None

    # biosample
    biosample_obj = exp.get("biosample_ontology", {})
    biosample = biosample_obj.get("term_name") if isinstance(biosample_obj, dict) else None

    # assay type
    assay = exp.get("assay_title")

    records.append({
        "accession": accession,
        "target": target_label,
        "biosample": biosample,
        "assay": assay
    })

df_histone = pd.DataFrame(records)



In [6]:
len(df_histone["biosample"].unique())

269

In [9]:
df_histone

,accession,target,biosample,assay
0,ENCSR000AMU,H3K4me1,fibroblast of lung,Histone ChIP-seq
1,ENCSR000APK,H3K4me2,osteoblast,Histone ChIP-seq
2,ENCSR000EXI,H3K4me3,Panc1,Histone ChIP-seq
3,ENCSR000DXF,H3K4me3,epithelial cell of proximal tubule,Histone ChIP-seq
4,ENCSR000DQP,H3K4me3,B cell,Histone ChIP-seq
...,...,...,...,...
4072,ENCSR000AKD,H3K27me3,GM12878,Histone ChIP-seq
4073,ENCSR668LDD,H3K4me3,K562,Histone ChIP-seq
4074,ENCSR424ZJH,H2AK5ac,IMR-90,Histone ChIP-seq
4075,ENCSR004ZWW,H4K8ac,IMR-90,Histone ChIP-seq


In [11]:
df[(df["target"] == "EP300") & (df["biosample"] == "T47D")]

,accession,target,biosample,assay_title,status
3883,ENCSR000BLM,EP300,T47D,TF ChIP-seq,released
